# LeetCode 220 — Contains Duplicate III bằng Python

Notebook này giảng **từ căn bản** một bài **Hard** rất hay để học cách "tìm nhanh một giá trị **gần** giá trị cho trước" trong một **cửa sổ trượt**.

Skeleton trên LeetCode (Python3, bản hiện hành):

```python
class Solution:
    def containsNearbyAlmostDuplicate(self, nums: List[int], indexDiff: int, valueDiff: int) -> bool:

```

> Ghi chú: bản cũ của đề đặt tên tham số là `k` và `t` (`containsNearbyAlmostDuplicate(self, nums, k, t)`). Bản hiện hành đổi thành `indexDiff` và `valueDiff` — ý nghĩa **y hệt**. Trong phần giảng, để công thức gọn, ta hay viết tắt **`k = indexDiff`** và **`t = valueDiff`**.

## Sau notebook này, bạn sẽ nắm được

1. **Ôn lại từ đầu** họ bài "Contains Duplicate": #217 (dùng `set`), #219 (dùng `dict` và **cửa sổ trượt**) — vì #220 chính là bước tiếp theo của hai bài này.
2. Đọc đúng **ba điều kiện** của đề #220 và tự kiểm tra bằng tay.
3. Công cụ Python cần dùng: `set`, `dict`, cửa sổ trượt, phép chia lấy phần nguyên `//` **với số âm**, **binary search** (tìm kiếm nhị phân) và module `bisect` (`bisect_left`, `bisect_right`, `insort`), sắp xếp `tuple`.
4. Bốn cách giải, đi từ trực giác đến tối ưu:
   - **Cách 1:** Brute force — so mọi cặp `i < j` có `j - i <= k`: `O(n·k)`.
   - **Cách 2:** Cửa sổ trượt + **list đã sắp xếp** + `bisect`: tìm kiếm `O(log k)`, nhưng chèn/xóa trong `list` tốn `O(k)` ⇒ tổng `O(n·k)` trên lý thuyết; ý tưởng này đạt `O(n log k)` nếu dùng **cây cân bằng**.
   - **Cách 3 (cách chính):** **Bucket** (chia "thùng") rộng `valueDiff + 1` — `O(n)`.
   - **Cách 4 (tham khảo):** Sắp xếp theo giá trị kèm chỉ số rồi quét — một góc nhìn khác, và vì sao nó **không** tốt bằng.
5. Vì sao mỗi cách đúng (**invariant** — "bất biến", điều luôn đúng sau mỗi bước), độ phức tạp, lỗi hay gặp, mẹo nhận dạng và bài tập tự luyện.

## 1. Đề bài (theo LeetCode)

You are given an integer array `nums` and two integers `indexDiff` and `valueDiff`.

Find a pair of indices `(i, j)` such that:

- `i != j`,
- `abs(i - j) <= indexDiff`.
- `abs(nums[i] - nums[j]) <= valueDiff`, and

Return `true` *if such pair exists or* `false` *otherwise*.

**Example 1**

```text
Input: nums = [1,2,3,1], indexDiff = 3, valueDiff = 0
Output: true
Explanation: We can choose (i, j) = (0, 3).
We satisfy the three conditions:
i != j --> 0 != 3
abs(i - j) <= indexDiff --> abs(0 - 3) <= 3
abs(nums[i] - nums[j]) <= valueDiff --> abs(1 - 1) <= 0
```

**Example 2**

```text
Input: nums = [1,5,9,1,5,9], indexDiff = 2, valueDiff = 3
Output: false
Explanation: After trying all the possible pairs (i, j), we cannot satisfy the three conditions, so we return false.
```

**Constraints**

- `2 <= nums.length <= 10^5`
- `-10^9 <= nums[i] <= 10^9`
- `1 <= indexDiff <= nums.length`
- `0 <= valueDiff <= 10^9`

### Dịch nhanh

- Cho mảng số nguyên `nums` và hai số `indexDiff`, `valueDiff`.
- Hỏi: có **hai vị trí khác nhau** `i`, `j` mà **gần nhau về chỉ số** (cách nhau không quá `indexDiff` ô) **và** **gần nhau về giá trị** (chênh lệch không quá `valueDiff`) hay không?
- Trả về `True`/`False` (trong Python viết hoa chữ đầu; `true`/`false` là cách viết của đề).
- `abs(...)` là **giá trị tuyệt đối**: `abs(-3) = 3`, `abs(3) = 3`. Nó đo "khoảng cách" giữa hai số, không quan tâm số nào lớn hơn.
- Lưu ý constraints: giá trị có thể **âm** (tới `-10^9`), `valueDiff` có thể bằng `0`, `indexDiff` có thể lớn bằng cả `n`.

## 2. Hiểu đề bằng tay

Tên bài nói lên tất cả: **Nearby** (gần về **chỉ số**) + **Almost Duplicate** (gần như trùng về **giá trị**).

| Điều kiện | Ý nghĩa | Với Ví dụ 1 (`k = 3`, `t = 0`) |
|---|---|---|
| `i != j` | hai vị trí **khác nhau** (không được ghép một phần tử với chính nó) | `0 != 3` |
| `abs(i - j) <= k` | gần nhau về **chỉ số** | `abs(0 - 3) = 3 <= 3` |
| `abs(nums[i] - nums[j]) <= t` | gần nhau về **giá trị** | `abs(1 - 1) = 0 <= 0` |

### Chỉ cần xét `i < j`

Vì `abs(i - j) = abs(j - i)` và `abs(a - b) = abs(b - a)`, cặp `(3, 0)` và cặp `(0, 3)` là **một**. Vậy ta chỉ cần xét các cặp có `i < j` — khi đó `abs(i - j)` chính là `j - i`.

### Liệt kê mọi cặp của Ví dụ 1

`nums = [1, 2, 3, 1]` có 4 phần tử ⇒ có `4·3/2 = 6` cặp `i < j`:

| Cặp | `j - i` | `<= 3`? | `abs(nums[i] - nums[j])` | `<= 0`? | Thỏa cả hai? |
|---|---:|:---:|---:|:---:|:---:|
| (0,1) | 1 | có | 1 | không | |
| (0,2) | 2 | có | 2 | không | |
| (0,3) | 3 | có | 0 | có | **THỎA** |
| (1,2) | 1 | có | 1 | không | |
| (1,3) | 2 | có | 1 | không | |
| (2,3) | 1 | có | 2 | không | |

Ô code dưới làm đúng việc này bằng hai vòng `for` lồng nhau (ta sẽ quay lại ý này ở Cách 1).

In [1]:
from typing import List   # LeetCode tự import sẵn; trong notebook ta phải tự import


def pairs_report(nums, k, t):
    """In MỌI cặp (i, j) với i < j và đánh dấu cặp thỏa cả hai điều kiện."""
    found = False
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            idx_ok = (j - i) <= k
            val_ok = abs(nums[i] - nums[j]) <= t
            mark = "<== THỎA" if idx_ok and val_ok else ""
            print(f"({i},{j}): j-i={j - i} {'<=' if idx_ok else '> '} {k} | "
                  f"|{nums[i]}-{nums[j]}|={abs(nums[i] - nums[j])} {'<=' if val_ok else '> '} {t} {mark}")
            found = found or (idx_ok and val_ok)
    print("Kết luận:", found)


pairs_report([1, 2, 3, 1], k=3, t=0)

(0,1): j-i=1 <= 3 | |1-2|=1 >  0 
(0,2): j-i=2 <= 3 | |1-3|=2 >  0 
(0,3): j-i=3 <= 3 | |1-1|=0 <= 0 <== THỎA
(1,2): j-i=1 <= 3 | |2-3|=1 >  0 
(1,3): j-i=2 <= 3 | |2-1|=1 >  0 
(2,3): j-i=1 <= 3 | |3-1|=2 >  0 
Kết luận: True


### Ví dụ 2 — vì sao `False`?

`nums = [1, 5, 9, 1, 5, 9]`, `k = 2`, `t = 3`. Có `6·5/2 = 15` cặp. Chạy ô dưới, để ý hai kiểu "suýt thỏa":

- Cặp `(0, 3)`: hai số `1` **bằng nhau** (chênh `0 <= 3`), nhưng chỉ số cách nhau `3 > 2` ⇒ **quá xa về chỉ số**.
- Cặp `(0, 1)`: chỉ số cách `1 <= 2`, nhưng `|1 - 5| = 4 > 3` ⇒ **quá xa về giá trị** (chỉ hụt đúng 1 đơn vị!).

Không cặp nào thỏa **đồng thời** cả hai ⇒ `False`.

### Vài edge case (trường hợp biên) nên nhớ ngay từ đầu

- `valueDiff = 0` ⇒ "gần về giá trị" nghĩa là **bằng nhau** ⇒ bài trở thành đúng bài #219 (sẽ ôn ở mục 4).
- `indexDiff = n` ⇒ mọi cặp đều đủ gần về chỉ số ⇒ chỉ còn hỏi "có hai phần tử nào chênh nhau `<= t` không?".
- Số âm: `nums = [-3, -1]`, `k = 1`, `t = 2` ⇒ `|-3 - (-1)| = 2 <= 2` ⇒ `True`.
- `nums.length >= 2` nên luôn có ít nhất một cặp để xét; không cần xử lý mảng rỗng.

In [2]:
pairs_report([1, 5, 9, 1, 5, 9], k=2, t=3)

(0,1): j-i=1 <= 2 | |1-5|=4 >  3 
(0,2): j-i=2 <= 2 | |1-9|=8 >  3 
(0,3): j-i=3 >  2 | |1-1|=0 <= 3 
(0,4): j-i=4 >  2 | |1-5|=4 >  3 
(0,5): j-i=5 >  2 | |1-9|=8 >  3 
(1,2): j-i=1 <= 2 | |5-9|=4 >  3 
(1,3): j-i=2 <= 2 | |5-1|=4 >  3 
(1,4): j-i=3 >  2 | |5-5|=0 <= 3 
(1,5): j-i=4 >  2 | |5-9|=4 >  3 
(2,3): j-i=1 <= 2 | |9-1|=8 >  3 
(2,4): j-i=2 <= 2 | |9-5|=4 >  3 
(2,5): j-i=3 >  2 | |9-9|=0 <= 3 
(3,4): j-i=1 <= 2 | |1-5|=4 >  3 
(3,5): j-i=2 <= 2 | |1-9|=8 >  3 
(4,5): j-i=1 <= 2 | |5-9|=4 >  3 
Kết luận: False


## 3. Ôn từ căn bản: LeetCode 217 — Contains Duplicate

> Given an integer array `nums`, return `true` if any value appears **at least twice** in the array, and return `false` if every element is distinct.

Không có điều kiện chỉ số, không có `t`: chỉ hỏi "có giá trị nào xuất hiện **từ 2 lần trở lên** không?".

### Công cụ: `set` (tập hợp)

- `set` giống một "cái túi" chứa các giá trị **không trùng nhau**. Thêm một giá trị đã có thì túi không đổi.
- `s.add(x)` — thêm `x`; `x in s` — hỏi "`x` có trong túi không?"; `s.remove(x)` — bỏ `x` (báo lỗi nếu không có); `s.discard(x)` — bỏ `x` nếu có (không báo lỗi).
- `x in s` chạy **trung bình `O(1)`** nhờ kỹ thuật **hashing** (băm): Python tính một "mã băm" của `x` để nhảy thẳng tới chỗ lưu, không phải dò từng phần tử như `list`.

### Lời giải #217: "kiểm tra trước, thêm sau"

Duyệt từng `x`. Nếu `x` **đã có** trong `seen` ⇒ ta vừa gặp lần thứ hai ⇒ `True`. Nếu chưa, thêm `x` vào `seen`. Hết mảng mà không trùng ⇒ `False`.

Thứ tự "kiểm tra **rồi mới** thêm" rất quan trọng: nếu thêm trước thì `x in seen` luôn đúng (vì vừa thêm chính nó). Ý này sẽ quay lại ở #220.

In [3]:
seen = set()
seen.add(3)
seen.add(7)
seen.add(3)                 # 3 đã có -> túi không đổi
print(seen)                 # {3, 7}
print(7 in seen, 5 in seen) # True False
seen.remove(7)
print(seen)                 # {3}
seen.discard(100)           # 100 không có -> discard không báo lỗi
print(len(set([1, 2, 3, 1])), "giá trị khác nhau trong [1, 2, 3, 1]")

{3, 7}
True False
{3}
3 giá trị khác nhau trong [1, 2, 3, 1]


In [4]:
def contains_duplicate_217(nums):
    seen = set()
    for x in nums:
        if x in seen:       # 1) kiểm tra trước
            return True
        seen.add(x)         # 2) thêm sau
    return False


assert contains_duplicate_217([1, 2, 3, 1]) == True
assert contains_duplicate_217([1, 2, 3, 4]) == False
assert contains_duplicate_217([1, 1, 1, 3, 3, 4, 3, 2, 4, 2]) == True
print("#217 PASS")

#217 PASS


## 4. Ôn LeetCode 219 — Contains Duplicate II

> Given an integer array `nums` and an integer `k`, return `true` if there are two **distinct indices** `i` and `j` in the array such that `nums[i] == nums[j]` and `abs(i - j) <= k`.

Đã có thêm **điều kiện chỉ số**, nhưng giá trị vẫn phải **bằng hẳn**. Nhìn kỹ: đây chính là #220 với `t = 0`.

### 4.1. Cách nghĩ đầu tiên: `dict` lưu "lần gặp gần nhất"

**`dict`** (từ điển) lưu các cặp **key → value**. Ở đây key là **giá trị** `x`, value là **chỉ số lần gần nhất** gặp `x`:

- `last[x] = i` — ghi (hoặc ghi đè) "gặp `x` lần gần nhất ở vị trí `i`".
- `x in last` — hỏi "đã từng gặp `x` chưa?" (trung bình `O(1)`).
- `last[x]` — đọc ra vị trí đã lưu; `del last[x]` — xóa key `x`.

Khi gặp `x` ở vị trí `i` mà `x in last`: lấy `j = last[x]`, nếu `i - j <= k` ⇒ `True`.

Để duyệt **cùng lúc** chỉ số và giá trị, ta dùng `for i, x in enumerate(nums)`: `enumerate` sinh ra từng cặp `(chỉ số, giá trị)`. Với `[5, 8, 5]` nó cho `(0, 5)`, `(1, 8)`, `(2, 5)` — mỗi vòng `i` nhận phần tử đầu, `x` nhận phần tử sau của cặp.

**Vì sao chỉ cần lần gần nhất?** Nếu `x` từng xuất hiện ở các vị trí `1, 4, 6` và giờ ta đứng ở `i = 8`, thì vị trí **6** cho khoảng cách nhỏ nhất (`8 - 6 = 2`). Nếu ngay cả vị trí gần nhất cũng quá xa thì các vị trí cũ hơn càng xa hơn.

Ví dụ của #219: `[1,2,3,1], k=3 → true`; `[1,0,1,1], k=1 → true`; `[1,2,3,1,2,3], k=2 → false`.

In [5]:
last = {}
last[5] = 0
last[8] = 1
last[5] = 3                  # gặp lại 5 ở vị trí 3 -> ghi đè
print(last)                  # {5: 3, 8: 1}
print(5 in last, last[5])    # True 3
del last[8]
print(last)                  # {5: 3}
print(list(enumerate([5, 8, 5])))   # [(0, 5), (1, 8), (2, 5)]


def contains_nearby_duplicate_219_dict(nums, k):
    last = {}
    for i, x in enumerate(nums):           # enumerate cho cả chỉ số i và giá trị x
        if x in last and i - last[x] <= k:
            return True
        last[x] = i                        # luôn cập nhật lần gặp gần nhất
    return False


assert contains_nearby_duplicate_219_dict([1, 2, 3, 1], 3) == True
assert contains_nearby_duplicate_219_dict([1, 0, 1, 1], 1) == True
assert contains_nearby_duplicate_219_dict([1, 2, 3, 1, 2, 3], 2) == False
print("#219 (dict) PASS")

{5: 3, 8: 1}
True 3
{5: 3}
[(0, 5), (1, 8), (2, 5)]
#219 (dict) PASS


### 4.2. Ý tưởng then chốt: **cửa sổ trượt** (sliding window)

Đứng ở vị trí `i`, những `j < i` **đủ gần** là những `j` thỏa `i - j <= k`, tức `j` nằm trong đoạn `[i - k, i - 1]`. Đó là **tối đa `k` phần tử ngay trước `i`** — ta gọi là **cửa sổ** của bước `i`.

Khi `i` tăng thêm 1, cửa sổ **trượt** sang phải đúng 1 ô:

- phần tử `nums[i]` (vừa xét xong) **đi vào** cửa sổ,
- phần tử `nums[i - k]` (giờ đã cách `i + 1` tới `k + 1` ô — quá xa) **đi ra** khỏi cửa sổ.

Ở những bước đầu (`i < k`), đoạn `[i - k, i - 1]` "lòi" ra ngoài mảng về bên trái, nên ta chặn bằng `max(0, i - k)`.

Chạy thử với `nums = [1, 2, 3, 1, 2, 3]`, `k = 2`:

In [6]:
nums = [1, 2, 3, 1, 2, 3]
k = 2
for i in range(len(nums)):
    start = max(0, i - k)
    where = f"nums[{start}..{i - 1}]" if i > start else "(chưa có phần tử nào)"
    print(f"i={i}, x={nums[i]}, cửa sổ = {where} = {nums[start:i]}")

i=0, x=1, cửa sổ = (chưa có phần tử nào) = []
i=1, x=2, cửa sổ = nums[0..0] = [1]
i=2, x=3, cửa sổ = nums[0..1] = [1, 2]
i=3, x=1, cửa sổ = nums[1..2] = [2, 3]
i=4, x=2, cửa sổ = nums[2..3] = [3, 1]
i=5, x=3, cửa sổ = nums[3..4] = [1, 2]


### 4.3. #219 bằng cửa sổ `set`

Thay vì lưu chỉ số, ta giữ một `set` tên `window` chứa **đúng các giá trị trong cửa sổ**. Mỗi bước `i`:

1. **Kiểm tra:** nếu `x in window` ⇒ có một `j` trong cửa sổ với `nums[j] == x` và `i - j <= k` ⇒ `True`.
2. **Thêm:** `window.add(x)`.
3. **Bỏ phần tử quá hạn:** nếu `i >= k`, bỏ `nums[i - k]`.

**Vì sao bỏ `nums[i - k]` ngay sau khi thêm?** Sau bước 2, `window` chứa `nums[i-k..i]` (`k + 1` phần tử). Bước kế tiếp `i + 1` cần cửa sổ `nums[(i+1)-k .. i] = nums[i-k+1 .. i]` — tức là phải bỏ đúng `nums[i - k]`. Ví dụ `k = 2`, `i = 2`: sau khi thêm, `window = {nums[0], nums[1], nums[2]}`; bỏ `nums[0]` còn `{nums[1], nums[2]}` — đúng cửa sổ của `i = 3`.

**Vì sao `remove` an toàn dù `set` không lưu trùng?** Nếu trong cửa sổ có hai giá trị bằng nhau thì ta đã trả `True` từ trước. Vậy khi chưa dừng, mọi giá trị trong cửa sổ **khác nhau** ⇒ mỗi giá trị trong `set` ứng với **đúng một** vị trí, và bỏ `nums[i - k]` là bỏ đúng phần tử đó. (Lập luận "đã dừng từ trước nên không thể trùng" này sẽ xuất hiện lại y hệt ở Cách 3.)

In [7]:
import random


def contains_nearby_duplicate_219_window(nums, k):
    window = set()
    for i, x in enumerate(nums):
        if x in window:                # 1) kiểm tra
            return True
        window.add(x)                  # 2) thêm
        if i >= k:
            window.remove(nums[i - k]) # 3) bỏ phần tử đã quá xa
    return False


assert contains_nearby_duplicate_219_window([1, 2, 3, 1], 3) == True
assert contains_nearby_duplicate_219_window([1, 0, 1, 1], 1) == True
assert contains_nearby_duplicate_219_window([1, 2, 3, 1, 2, 3], 2) == False

random.seed(219)
for _ in range(2000):
    arr = [random.randint(0, 5) for _ in range(random.randint(2, 10))]
    kk = random.randint(1, len(arr))
    assert contains_nearby_duplicate_219_window(arr, kk) == contains_nearby_duplicate_219_dict(arr, kk)
print("#219 (cửa sổ set) PASS, khớp bản dict trên 2000 test ngẫu nhiên")

#219 (cửa sổ set) PASS, khớp bản dict trên 2000 test ngẫu nhiên


## 5. Từ #219 sang #220: chỗ khác nhau duy nhất

Ở #219 ta hỏi cửa sổ: "**có đúng giá trị `x` không?**" — `set` trả lời trong `O(1)`.

Ở #220 ta phải hỏi: "**có giá trị nào chênh với `x` không quá `t` không?**" Viết lại điều kiện giá trị thành một **đoạn**:

$$|x - y| \le t \iff x - t \le y \le x + t$$

Ví dụ `x = 5`, `t = 3` ⇒ cần tìm `y` trong đoạn `[2, 8]`. `y = 2` hay `y = 8` đều được (chênh đúng 3), `y = 1` hay `y = 9` thì không.

Vậy câu hỏi mỗi bước là: "**cửa sổ có phần tử nào rơi vào đoạn `[x - t, x + t]` không?**"

`set` **không** trả lời trực tiếp được câu này. Cách "ngây thơ" là hỏi `set` lần lượt từng giá trị `x - t, x - t + 1, ..., x + t` — tức `2t + 1` lần hỏi. Với `t = 3` thì chỉ 7 lần, nhưng `t` có thể tới `10^9` ⇒ **2 tỉ lần hỏi cho mỗi phần tử**. Không thể dùng.

Từ đây có ba hướng, cũng là lộ trình của notebook:

1. **Không dùng cấu trúc gì cả**, so `x` với từng phần tử trong cửa sổ ⇒ **Cách 1** (brute force).
2. Giữ cửa sổ **đã sắp xếp** để tìm phần tử **gần `x` nhất** thật nhanh ⇒ **Cách 2** (`bisect`).
3. **Gom giá trị thành từng "khoảng"** để một lần tra `dict` trả lời cho cả một khoảng ⇒ **Cách 3** (bucket).

In [8]:
x, t = 5, 3
print("Đoạn cần tìm:", [x - t, x + t])
for y in [1, 2, 5, 8, 9]:
    print(f"y={y}: |x-y|={abs(x - y)} <= {t}? {str(abs(x - y) <= t):<5} | {x - t} <= y <= {x + t}? {x - t <= y <= x + t}")


def window_has_close_naive(window_set, x, t):
    """Hỏi set 2t+1 lần: y = x-t, x-t+1, ..., x+t."""
    for y in range(x - t, x + t + 1):
        if y in window_set:
            return True
    return False


print(window_has_close_naive({1, 9}, 5, 3))   # False: 1 và 9 đều nằm ngoài [2, 8]
print(window_has_close_naive({1, 7}, 5, 3))   # True: 7 nằm trong [2, 8]
t_big = 10**9
print("Với t = 10^9, mỗi bước phải hỏi set", 2 * t_big + 1, "lần -> quá chậm!")

Đoạn cần tìm: [2, 8]
y=1: |x-y|=4 <= 3? False | 2 <= y <= 8? False
y=2: |x-y|=3 <= 3? True  | 2 <= y <= 8? True
y=5: |x-y|=0 <= 3? True  | 2 <= y <= 8? True
y=8: |x-y|=3 <= 3? True  | 2 <= y <= 8? True
y=9: |x-y|=4 <= 3? False | 2 <= y <= 8? False
False
True
Với t = 10^9, mỗi bước phải hỏi set 2000000001 lần -> quá chậm!


## 6. Cách 1 — Brute force: so mọi cặp gần nhau về chỉ số

### Ý tưởng

Cách người mới tự nghĩ ra: với mỗi `i`, thử mọi `j` ở **bên phải** `i` mà còn trong tầm `k` (tức `i < j <= i + k`), kiểm tra `abs(nums[i] - nums[j]) <= t`.

So với `pairs_report` (thử **cả 15 cặp** rồi mới lọc), ta tiết kiệm hơn một chút: không bao giờ xét cặp quá xa về chỉ số.

### Cận của vòng `j`

`j` chạy từ `i + 1` đến `i + k`, nhưng không được vượt chỉ số cuối `n - 1`. Nên cận trên là `min(i + k, n - 1)`, và vì `range` **không lấy** cận trên nên ta viết:

```python
for j in range(i + 1, min(i + k, n - 1) + 1):
```

Ví dụ `n = 6`, `k = 2`:

| `i` | `min(i + k, n - 1)` | các `j` được xét |
|---:|---:|---|
| 0 | `min(2, 5) = 2` | 1, 2 |
| 3 | `min(5, 5) = 5` | 4, 5 |
| 4 | `min(6, 5) = 5` | 5 |
| 5 | `min(7, 5) = 5` | (không có — `range(6, 6)` rỗng) |

Ngoài hai ví dụ của đề, ta dùng thêm một **ví dụ xuyên suốt** có số âm: `nums = [8, -4, 3, 9, -2, 0]`, `k = 2`, `t = 2`. Đáp án là `True` nhờ cặp `(4, 5)`: `|-2 - 0| = 2 <= 2` và `5 - 4 = 1 <= 2`.

In [9]:
def almost_dup_bruteforce(nums: List[int], indexDiff: int, valueDiff: int) -> bool:
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, min(i + indexDiff, n - 1) + 1):
            if abs(nums[i] - nums[j]) <= valueDiff:
                return True
    return False


print(almost_dup_bruteforce([1, 2, 3, 1], 3, 0))          # True
print(almost_dup_bruteforce([1, 5, 9, 1, 5, 9], 2, 3))    # False
print(almost_dup_bruteforce([8, -4, 3, 9, -2, 0], 2, 2))  # True (cặp (4, 5))
print(almost_dup_bruteforce([-3, -1], 1, 2))              # True

True
False
True
True


### Cách 1 tốn bao nhiêu công?

Gọi `n = len(nums)`, `k = indexDiff`. Mỗi `i` xét **tối đa `k`** giá trị `j` ⇒ tổng số phép so sánh **tối đa khoảng `n·k`**.

- **Time:** `O(n·k)`. Trường hợp xấu nhất `k = n` ⇒ `O(n²)`: khoảng `n(n-1)/2 ≈ 5·10^9` phép so sánh khi `n = 10^5`.
- **Space:** `O(1)` (chỉ vài biến đếm).
- Trường hợp **xấu nhất là khi đáp án `False`**: ta phải thử **hết** mới dám kết luận. Khi có đáp án `True`, vòng lặp có thể dừng sớm.

Python chạy cỡ vài chục triệu phép so sánh đơn giản mỗi giây ⇒ `5·10^9` phép mất **hàng trăm giây** ⇒ chắc chắn **TLE** (Time Limit Exceeded — quá thời gian).

### Nút thắt nằm ở đâu?

Với mỗi `x = nums[i]`, ta so `x` với **từng** phần tử trong cửa sổ (tới `k` phần tử). Nhưng thật ra ta chỉ cần biết: **phần tử nào trong cửa sổ gần `x` nhất?** Nếu phần tử gần nhất còn chênh hơn `t` thì những phần tử khác càng chênh hơn. Câu hỏi "ai gần `x` nhất" trả lời được rất nhanh nếu cửa sổ **đã được sắp xếp** — đó là bước đột phá ở mục 7.

Ô dưới đếm số phép so sánh của Cách 1 (công thức chính xác) và đo thời gian thật trên một input `False` cỡ `n = k = 3000` để ước lượng cho `n = k = 10^5`.

In [10]:
import time


def count_bruteforce_comparisons(n, k):
    """Số cặp (i, j) mà Cách 1 phải xét khi đáp án False (không dừng sớm)."""
    return sum(min(i + k, n - 1) - i for i in range(n))


for n, k in [(6, 2), (1000, 10), (1000, 1000), (10**5, 10**5)]:
    print(f"n={n:>6}, k={k:>6}: {count_bruteforce_comparisons(n, k):>13,} phép so sánh")

random.seed(1)
arr = random.sample(range(-10**9, 10**9), 3000)   # 3000 số khác nhau, t = 0 -> đáp án False
start = time.perf_counter()
print("Kết quả:", almost_dup_bruteforce(arr, 3000, 0))
elapsed = time.perf_counter() - start
per_cmp = elapsed / count_bruteforce_comparisons(3000, 3000)
print(f"n = k = 3000 mất {elapsed:.2f} giây")
print(f"Ước lượng n = k = 10^5: khoảng {per_cmp * count_bruteforce_comparisons(10**5, 10**5):.0f} giây -> TLE")

n=     6, k=     2:             9 phép so sánh
n=  1000, k=    10:         9,945 phép so sánh
n=  1000, k=  1000:       499,500 phép so sánh
n=100000, k=100000: 4,999,950,000 phép so sánh


Kết quả: False
n = k = 3000 mất 0.32 giây
Ước lượng n = k = 10^5: khoảng 361 giây -> TLE


## 7. Bước đột phá 1: chỉ cần hỏi **một** phần tử của cửa sổ

Giả sử cửa sổ đang chứa `{12, 1, 7, 20}`, sắp xếp lại thành `W = [1, 7, 12, 20]`.

Gặp `x = 10`, `t = 2` ⇒ cần tìm phần tử trong đoạn `[8, 12]`.

**Mẹo:** tìm **phần tử nhỏ nhất của `W` mà `>= x - t`** — gọi nó là `c` (trong sách hay gọi là **ceiling** — "trần" — của `x - t`).

- Với `x - t = 8`: các phần tử `>= 8` là `12, 20` ⇒ `c = 12`. Vì `c = 12 <= x + t = 12` ⇒ **có** phần tử trong đoạn ⇒ `True`.
- Với `x = 4`, `t = 2` (đoạn `[2, 6]`): các phần tử `>= 2` là `7, 12, 20` ⇒ `c = 7 > 6` ⇒ **không có** ⇒ `False`. Đúng vậy: `1` quá nhỏ, `7` trở đi quá lớn.

### Vì sao chỉ cần kiểm tra `c`?

Gọi đoạn cần tìm là `[x - t, x + t]`.

- Nếu **có** một `y` trong cửa sổ với `x - t <= y <= x + t`: vì `y >= x - t`, nên `c` tồn tại và `c <= y` (do `c` là số **nhỏ nhất** `>= x - t`). Suy ra `x - t <= c <= y <= x + t` ⇒ `c` cũng nằm trong đoạn.
- Ngược lại, nếu `c` nằm trong đoạn thì hiển nhiên ta có phần tử cần tìm.

Kết luận: **"có phần tử trong đoạn" ⇔ "`c` tồn tại và `c <= x + t`"**. Chỉ cần tìm ra `c` — một phần tử duy nhất — thay vì so với cả `k` phần tử.

Ô dưới kiểm chứng bằng cách tìm `c` "thủ công" (duyệt từ trái sang trên list đã sắp xếp). Mục 8 sẽ thay việc duyệt này bằng **binary search** để nhanh hơn nhiều.

In [11]:
def ceiling_linear(sorted_w, v):
    """Phần tử nhỏ nhất của sorted_w mà >= v (duyệt tuần tự). Không có thì trả None."""
    for y in sorted_w:
        if y >= v:
            return y
    return None


W = sorted([12, 1, 7, 20])
print("W =", W)
for x, t in [(10, 2), (4, 2), (25, 3), (0, 1)]:
    c = ceiling_linear(W, x - t)
    ok = c is not None and c <= x + t
    print(f"x={x:>2}, t={t}: đoạn [{x - t}, {x + t}], c = ceiling({x - t}) = {c} -> có phần tử gần? {ok}")

W = [1, 7, 12, 20]
x=10, t=2: đoạn [8, 12], c = ceiling(8) = 12 -> có phần tử gần? True
x= 4, t=2: đoạn [2, 6], c = ceiling(2) = 7 -> có phần tử gần? False
x=25, t=3: đoạn [22, 28], c = ceiling(22) = None -> có phần tử gần? False
x= 0, t=1: đoạn [-1, 1], c = ceiling(-1) = 1 -> có phần tử gần? True


## 8. Nền tảng: binary search và module `bisect`

### 8.1. Binary search (tìm kiếm nhị phân) là gì?

Trò chơi đoán số từ 1 đến 100, mỗi lần được trả lời "lớn hơn / nhỏ hơn": đoán 50 ⇒ bỏ được một nửa; đoán 25 hoặc 75 ⇒ bỏ tiếp một nửa... Sau tối đa **7** lần là ra, vì `2^7 = 128 >= 100`.

Trên một **list đã sắp xếp** dài `k`, binary search tìm vị trí chỉ mất khoảng `log₂(k)` bước: `k = 10^5` ⇒ chỉ **17 bước**, thay vì duyệt cả `10^5` phần tử.

### 8.2. `bisect_left` và `bisect_right`

Module chuẩn `bisect` làm sẵn binary search cho ta:

- `bisect.bisect_left(a, v)` = **chỉ số nhỏ nhất `p` mà `a[p] >= v`**. Mọi phần tử đứng trước `p` đều `< v`. Đây cũng là vị trí để **chèn `v`** sao cho `a` vẫn sắp xếp, đặt `v` **bên trái** các số bằng nó.
- `bisect.bisect_right(a, v)` = chỉ số nhỏ nhất `p` mà `a[p] > v` (chèn `v` **bên phải** các số bằng nó).
- Nếu không có phần tử nào `>= v` thì `bisect_left` trả về `len(a)` — tức "chèn vào cuối".

Với `a = [1, 3, 3, 7, 9]` (chỉ số `0..4`):

| `v` | `bisect_left` | `bisect_right` | Ý nghĩa của `bisect_left` |
|---:|---:|---:|---|
| 0 | 0 | 0 | mọi phần tử đều `>= 0` ⇒ ceiling là `a[0] = 1` |
| 3 | 1 | 3 | số `3` đầu tiên ở chỉ số 1 |
| 4 | 3 | 3 | ceiling của 4 là `a[3] = 7` |
| 9 | 4 | 5 | ceiling của 9 là chính `a[4] = 9` |
| 10 | 5 | 5 | `= len(a)` ⇒ **không có** phần tử `>= 10` |

Liên hệ với mục 7: **`c = a[bisect_left(a, x - t)]`** (nếu chỉ số đó `< len(a)`).

In [12]:
import bisect

a = [1, 3, 3, 7, 9]
for v in [0, 3, 4, 9, 10]:
    p = bisect.bisect_left(a, v)
    ceiling = a[p] if p < len(a) else None
    print(f"v={v:>2}: bisect_left={p}, bisect_right={bisect.bisect_right(a, v)}, ceiling={ceiling}")

v= 0: bisect_left=0, bisect_right=0, ceiling=1
v= 3: bisect_left=1, bisect_right=3, ceiling=3
v= 4: bisect_left=3, bisect_right=3, ceiling=7
v= 9: bisect_left=4, bisect_right=5, ceiling=9
v=10: bisect_left=5, bisect_right=5, ceiling=None


### 8.3. Tự cài `bisect_left` để hiểu bên trong

Ta giữ hai biên `lo`, `hi` với **bất biến**:

- mọi phần tử ở chỉ số `< lo` đều `< v` (chắc chắn **không** phải đáp án),
- mọi phần tử ở chỉ số `>= hi` đều `>= v` (đáp án nằm ở `hi` hoặc bên trái nó).

Ban đầu `lo = 0`, `hi = len(a)` (chưa biết gì). Mỗi vòng lấy `mid = (lo + hi) // 2`:

- nếu `a[mid] < v` ⇒ `mid` và mọi thứ bên trái nó đều `< v` ⇒ `lo = mid + 1`;
- ngược lại `a[mid] >= v` ⇒ đáp án ở `mid` hoặc bên trái ⇒ `hi = mid`.

Khi `lo == hi`, khoảng "chưa biết" rỗng ⇒ `lo` là đáp án.

Tính tay với `a = [1, 3, 3, 7, 9]`, `v = 4`:

| Vòng | `lo` | `hi` | `mid` | `a[mid]` | So với 4 | Cập nhật |
|---:|---:|---:|---:|---:|---|---|
| 1 | 0 | 5 | 2 | 3 | `3 < 4` | `lo = 3` |
| 2 | 3 | 5 | 4 | 9 | `9 >= 4` | `hi = 4` |
| 3 | 3 | 4 | 3 | 7 | `7 >= 4` | `hi = 3` |
| hết | 3 | 3 | | | | trả về **3** |

In [13]:
def my_bisect_left(a, v, show=False):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if show:
            print(f"  lo={lo}, hi={hi}, mid={mid}, a[mid]={a[mid]}", "< v" if a[mid] < v else ">= v")
        if a[mid] < v:
            lo = mid + 1
        else:
            hi = mid
    return lo


print("Trace my_bisect_left([1, 3, 3, 7, 9], 4):")
print("Kết quả:", my_bisect_left([1, 3, 3, 7, 9], 4, show=True))

random.seed(8)
for _ in range(3000):
    arr = sorted(random.randint(-10, 10) for _ in range(random.randint(0, 8)))
    v = random.randint(-12, 12)
    assert my_bisect_left(arr, v) == bisect.bisect_left(arr, v)
print("my_bisect_left khớp bisect.bisect_left trên 3000 test ngẫu nhiên")

Trace my_bisect_left([1, 3, 3, 7, 9], 4):
  lo=0, hi=5, mid=2, a[mid]=3 < v
  lo=3, hi=5, mid=4, a[mid]=9 >= v
  lo=3, hi=4, mid=3, a[mid]=7 >= v
Kết quả: 3
my_bisect_left khớp bisect.bisect_left trên 3000 test ngẫu nhiên


### 8.4. Chèn và xóa trên list đã sắp xếp — và cái giá phải trả

- **Chèn:** `bisect.insort(a, v)` tìm vị trí bằng binary search rồi gọi `a.insert(vị trí, v)`, nên `a` **vẫn sắp xếp**. (`insort` là tên gọi tắt của `insort_right`: chèn bên phải các số bằng `v`.)
- **Xóa một bản sao của `v`** (biết chắc `v` có trong `a`): `p = bisect.bisect_left(a, v)` rồi `a.pop(p)`. Với số trùng, ta xóa bản sao nào cũng như nhau vì chúng **cùng giá trị**.

**Cái giá:** tìm vị trí chỉ `O(log k)`, nhưng `list` trong Python là một dãy ô nhớ **liên tiếp**. Chèn vào giữa ⇒ mọi phần tử phía sau phải **dịch sang phải 1 ô**; xóa ở giữa ⇒ phía sau dịch sang trái. Ví dụ chèn `4` vào `[1, 3, 3, 7, 9]`: hai số `7, 9` phải dịch. Với list dài `k`, có thể phải dịch tới `k` phần tử ⇒ **`insert`/`pop` ở giữa tốn `O(k)`**.

> Muốn cả tìm, chèn, xóa đều `O(log k)` cần **cây nhị phân tìm kiếm cân bằng** (balanced BST — ví dụ cây đỏ-đen): Java có `TreeSet`, C++ có `std::set`/`std::multiset`. Python **không có sẵn** trong thư viện chuẩn; trên LeetCode có thư viện ngoài `sortedcontainers.SortedList` làm việc tương tự.

In [14]:
a = [1, 3, 3, 7, 9]
p = bisect.bisect_left(a, 4)
print("Chèn 4 vào vị trí", p, "-> phải dịch", len(a) - p, "phần tử:", a[p:])
bisect.insort(a, 4)
print("Sau insort(4):", a)
bisect.insort(a, 3)
print("Sau insort(3):", a)        # [1, 3, 3, 3, 4, 7, 9]
p = bisect.bisect_left(a, 3)
a.pop(p)
print("Xóa một số 3 :", a)        # [1, 3, 3, 4, 7, 9] - vẫn sắp xếp

Chèn 4 vào vị trí 3 -> phải dịch 2 phần tử: [7, 9]
Sau insort(4): [1, 3, 3, 4, 7, 9]
Sau insort(3): [1, 3, 3, 3, 4, 7, 9]
Xóa một số 3 : [1, 3, 3, 4, 7, 9]


### 8.5. Ghép lại: hỏi "có phần tử gần `x` không?" bằng `bisect`

```python
pos = bisect.bisect_left(window, x - t)        # vị trí của c = ceiling(x - t)
return pos < len(window) and window[pos] <= x + t
```

- `pos < len(window)` kiểm tra `c` **có tồn tại** không. Nếu `pos == len(window)` thì không phần tử nào `>= x - t`.
- Toán tử `and` của Python **dừng sớm** (short-circuit): nếu vế trái `False` thì vế phải **không được tính** ⇒ không bao giờ đọc `window[len(window)]` (sẽ gây `IndexError`).

Ô dưới so hàm mới với `ceiling_linear` ở mục 7 trên nhiều input ngẫu nhiên.

In [15]:
def has_close_in_sorted(window, x, t):
    """window đã sắp xếp. Có phần tử y nào với |x - y| <= t không? Tốn O(log len(window))."""
    pos = bisect.bisect_left(window, x - t)
    return pos < len(window) and window[pos] <= x + t


W = [1, 7, 12, 20]
for x, t in [(10, 2), (4, 2), (25, 3), (0, 1)]:
    print(f"x={x:>2}, t={t}: {has_close_in_sorted(W, x, t)}")

random.seed(85)
for _ in range(3000):
    w = sorted(random.randint(-20, 20) for _ in range(random.randint(0, 6)))
    x, t = random.randint(-25, 25), random.randint(0, 5)
    c = ceiling_linear(w, x - t)
    assert has_close_in_sorted(w, x, t) == (c is not None and c <= x + t)
print("has_close_in_sorted khớp cách duyệt tuần tự trên 3000 test")

x=10, t=2: True
x= 4, t=2: False
x=25, t=3: False
x= 0, t=1: True
has_close_in_sorted khớp cách duyệt tuần tự trên 3000 test


## 9. Cách 2 — Cửa sổ trượt + list đã sắp xếp + `bisect`

Ghép khung cửa sổ trượt của #219 (mục 4.3) với truy vấn `has_close_in_sorted` (mục 8.5). `window` bây giờ là một **list luôn được sắp xếp** chứa các giá trị của cửa sổ.

Với mỗi `i`, `x = nums[i]`:

1. **Hỏi:** `pos = bisect_left(window, x - t)`; nếu `pos < len(window)` và `window[pos] <= x + t` ⇒ `True`.
2. **Thêm:** `bisect.insort(window, x)`.
3. **Bỏ phần tử quá hạn:** nếu `i >= k`, xóa một bản sao của `nums[i - k]` khỏi `window`.

**Bất biến:** ở đầu bước `i`, `window` = các giá trị `nums[max(0, i-k) .. i-1]` **đã sắp xếp**. Bước 2 và 3 biến nó thành `nums[i-k+1 .. i]` — đúng cửa sổ của bước `i + 1` (lập luận y hệt mục 4.3).

**Đừng đảo bước 1 và 2:** nếu thêm `x` trước rồi mới hỏi, `x` sẽ tìm thấy **chính nó** (`|x - x| = 0 <= t`) ⇒ luôn trả `True`, vi phạm điều kiện `i != j`.

In [16]:
def almost_dup_bisect(nums: List[int], indexDiff: int, valueDiff: int) -> bool:
    k, t = indexDiff, valueDiff
    window = []                                   # luôn sắp xếp tăng dần
    for i, x in enumerate(nums):
        pos = bisect.bisect_left(window, x - t)   # 1) c = ceiling(x - t)
        if pos < len(window) and window[pos] <= x + t:
            return True
        bisect.insort(window, x)                  # 2) thêm x, vẫn giữ thứ tự
        if i >= k:                                # 3) bỏ nums[i - k]
            window.pop(bisect.bisect_left(window, nums[i - k]))
    return False


print(almost_dup_bisect([1, 2, 3, 1], 3, 0))          # True
print(almost_dup_bisect([1, 5, 9, 1, 5, 9], 2, 3))    # False
print(almost_dup_bisect([8, -4, 3, 9, -2, 0], 2, 2))  # True

True
False
True


### Trace Cách 2 với ví dụ xuyên suốt `nums = [8, -4, 3, 9, -2, 0]`, `k = 2`, `t = 2`

| `i` | `x` | `window` trước | Đoạn `[x-t, x+t]` | `pos` | `window[pos]` | Quyết định | Cập nhật |
|---:|---:|---|---|---:|---|---|---|
| 0 | 8 | `[]` | `[6, 10]` | 0 | không có (`pos == len`) | chưa | insort ⇒ `[8]` |
| 1 | -4 | `[8]` | `[-6, -2]` | 0 | `8 > -2` | chưa | insort ⇒ `[-4, 8]` |
| 2 | 3 | `[-4, 8]` | `[1, 5]` | 1 | `8 > 5` | chưa | insort ⇒ `[-4, 3, 8]`, xóa `nums[0]=8` ⇒ `[-4, 3]` |
| 3 | 9 | `[-4, 3]` | `[7, 11]` | 2 | không có | chưa | insort ⇒ `[-4, 3, 9]`, xóa `nums[1]=-4` ⇒ `[3, 9]` |
| 4 | -2 | `[3, 9]` | `[-4, 0]` | 0 | `3 > 0` | chưa | insort ⇒ `[-2, 3, 9]`, xóa `nums[2]=3` ⇒ `[-2, 9]` |
| 5 | 0 | `[-2, 9]` | `[-2, 2]` | 0 | `-2 <= 2` | **True** | |

Hai điểm đáng chú ý:

- **Bước `i = 3`** cho thấy vì sao phải xóa: nếu quên xóa `8`, `window` sẽ là `[-4, 3, 8]`, ceiling của `7` là `8 <= 11` ⇒ trả `True` **sai** — vì `8` ở chỉ số 0, `9` ở chỉ số 3, cách nhau `3 > k = 2`.
- **Bước `i = 5`**: ceiling của `x - t = -2` là chính `-2` (ở chỉ số 4 trong `nums`) ⇒ cặp `(4, 5)`, khớp với brute force.

In [17]:
def trace_bisect(nums, indexDiff, valueDiff):
    k, t = indexDiff, valueDiff
    window = []
    print(f"nums={nums}, k={k}, t={t}")
    for i, x in enumerate(nums):
        lo, hi = x - t, x + t
        pos = bisect.bisect_left(window, lo)
        cand = window[pos] if pos < len(window) else None
        head = f"i={i} x={x:>2} window={str(window):<9} đoạn=[{lo},{hi}] pos={pos}"
        if cand is not None and cand <= hi:
            print(head, f"window[{pos}]={cand} <= {hi} -> True")
            return True
        why = "pos==len (không có ứng viên)" if cand is None else f"window[{pos}]={cand} > {hi}"
        print(head, why)
        bisect.insort(window, x)
        if i >= k:
            old = nums[i - k]
            window.pop(bisect.bisect_left(window, old))
            print(f"      insort {x}, xóa nums[{i - k}]={old} -> {window}")
        else:
            print(f"      insort {x} -> {window}")
    print("Hết mảng -> False")
    return False


result = trace_bisect([8, -4, 3, 9, -2, 0], 2, 2)

nums=[8, -4, 3, 9, -2, 0], k=2, t=2
i=0 x= 8 window=[]        đoạn=[6,10] pos=0 pos==len (không có ứng viên)
      insort 8 -> [8]
i=1 x=-4 window=[8]       đoạn=[-6,-2] pos=0 window[0]=8 > -2
      insort -4 -> [-4, 8]
i=2 x= 3 window=[-4, 8]   đoạn=[1,5] pos=1 window[1]=8 > 5
      insort 3, xóa nums[0]=8 -> [-4, 3]
i=3 x= 9 window=[-4, 3]   đoạn=[7,11] pos=2 pos==len (không có ứng viên)
      insort 9, xóa nums[1]=-4 -> [3, 9]
i=4 x=-2 window=[3, 9]    đoạn=[-4,0] pos=0 window[0]=3 > 0
      insort -2, xóa nums[2]=3 -> [-2, 9]
i=5 x= 0 window=[-2, 9]   đoạn=[-2,2] pos=0 window[0]=-2 <= 2 -> True


### Độ phức tạp Cách 2 — nói cho thật chính xác

Gọi `n = len(nums)`, `k = indexDiff`. `window` có tối đa `k` phần tử (và không quá `n`).

| Thao tác mỗi bước | Chi phí |
|---|---|
| `bisect_left` (tìm ceiling) | `O(log k)` |
| `insort` = tìm vị trí `O(log k)` + `list.insert` **dịch phần tử** | `O(k)` |
| `pop` ở giữa list (dịch phần tử) | `O(k)` |

- **Time:** `O(n·k)` trong trường hợp xấu nhất **trên lý thuyết** — vì chèn/xóa giữa `list` phải dịch phần tử. Nghe thì "không hơn" brute force, **nhưng**: việc dịch phần tử được làm bằng một lệnh chép bộ nhớ khối (`memmove`) viết bằng C, nhanh hơn vòng lặp Python hàng trăm lần. Ở mục 14 ta sẽ đo: Cách 2 xử lý `n = k = 10^5` chỉ mất cỡ dưới một giây, trong khi Cách 1 cần hàng trăm giây.
- **Ý tưởng** của Cách 2 là `O(n log k)`: chỉ cần thay `list` bằng một cấu trúc mà chèn/xóa cũng `O(log k)` (cây cân bằng).
- **Space:** `O(min(n, k))` cho `window`.

Phiên bản dùng `SortedList` (chạy được trên LeetCode vì LeetCode cài sẵn `sortedcontainers`; **không** thuộc thư viện chuẩn nên notebook này không chạy nó):

```python
from sortedcontainers import SortedList

class Solution:
    def containsNearbyAlmostDuplicate(self, nums: List[int], indexDiff: int, valueDiff: int) -> bool:
        window = SortedList()
        for i, x in enumerate(nums):
            pos = window.bisect_left(x - valueDiff)
            if pos < len(window) and window[pos] <= x + valueDiff:
                return True
            window.add(x)                         # xấp xỉ O(log k)
            if i >= indexDiff:
                window.remove(nums[i - indexDiff])
        return False
```

Trong Java/C++ ý tưởng y hệt: `TreeSet.ceiling(x - t)` hoặc `set.lower_bound(x - t)`.

**Câu hỏi tiếp theo:** ở #219, mỗi bước chỉ tốn `O(1)` nhờ hashing. Có cách nào **không cần sắp xếp** mà vẫn trả lời được "có ai gần `x` không" trong `O(1)`? Có — đó là Cách 3.

## 10. Cách 3 (cách chính) — Bucket: "băm theo khoảng" thay vì "băm theo giá trị"

### Ý tưởng bằng đời thường

Muốn biết trong lớp có hai bạn **chênh nhau không quá 9 tuổi** không, ta có thể chia mọi người theo **thập niên**: `tuổi // 10` (0–9 tuổi ⇒ nhóm 0, 10–19 ⇒ nhóm 1, 20–29 ⇒ nhóm 2, ...). Hai người **cùng nhóm** chắc chắn chênh không quá 9 tuổi. Hai người ở **hai nhóm kề nhau** (ví dụ 19 và 20 tuổi) có thể chênh ít, có thể chênh nhiều — phải so thật. Hai người cách nhau **từ 2 nhóm trở lên** (ví dụ nhóm 1 và nhóm 3) chắc chắn chênh hơn 9 tuổi.

### Áp dụng vào bài

Chia trục số thành các **thùng** (bucket) liên tiếp, mỗi thùng rộng đúng **`w = t + 1`** giá trị. Thùng của số `x` là:

$$\text{id}(x) = x \;//\; w$$

Ví dụ `t = 2` ⇒ `w = 3`: thùng 0 = `{0, 1, 2}`, thùng 1 = `{3, 4, 5}`, thùng -1 = `{-3, -2, -1}`, ... Tổng quát, **thùng `b` chứa đúng các số trong đoạn `[b·w, b·w + w - 1]`**.

Ô dưới in các thùng cho `x` từ `-9` đến `8`.

In [18]:
t = 2
w = t + 1
groups = {}                         # id thùng -> danh sách các số rơi vào thùng đó
for x in range(-9, 9):
    b = x // w
    if b not in groups:
        groups[b] = []
    groups[b].append(x)

for b, xs in groups.items():
    print(f"thùng {b:>2}: {xs}   đoạn [{b * w}, {b * w + w - 1}]")

thùng -3: [-9, -8, -7]   đoạn [-9, -7]
thùng -2: [-6, -5, -4]   đoạn [-6, -4]
thùng -1: [-3, -2, -1]   đoạn [-3, -1]
thùng  0: [0, 1, 2]   đoạn [0, 2]
thùng  1: [3, 4, 5]   đoạn [3, 5]
thùng  2: [6, 7, 8]   đoạn [6, 8]


### 10.1. Phép chia `//` với số âm — chỗ dễ sai nhất của Cách 3

Trong Python, `//` là **chia rồi làm tròn xuống** (về phía âm vô cùng, gọi là *floor division*):

| Phép tính | Chia thật | Làm tròn **xuống** (`//`) | Làm tròn **về 0** (`int(x / w)`, kiểu C++/Java) |
|---|---:|---:|---:|
| `7 // 3` | 2.33 | **2** | 2 |
| `-1 // 3` | -0.33 | **-1** | 0 |
| `-3 // 3` | -1 | **-1** | -1 |
| `-4 // 3` | -1.33 | **-2** | -1 |
| `-7 // 3` | -2.33 | **-3** | -2 |

Với số dương hai cách giống nhau; với số âm thì **khác**. Vì sao `//` mới đúng cho bài này?

- Dùng `//`: thùng 0 = `{0, 1, 2}`, thùng -1 = `{-3, -2, -1}` — **mỗi thùng rộng đúng 3**.
- Dùng làm tròn về 0: cả `-2, -1, 0, 1, 2` đều có id `0` ⇒ "thùng 0" rộng **5** giá trị. Khi đó hai số "cùng thùng" như `-2` và `2` lại chênh nhau `4 > t = 2` ⇒ thuật toán sẽ trả `True` **sai** (minh họa ở mục Lỗi thường gặp).

> Trong C++/Java, phép `/` trên số nguyên làm tròn về 0, nên phải tự sửa: `id = x >= 0 ? x / w : (x + 1) / w - 1` (hoặc `Math.floorDiv(x, w)` trong Java). Python không cần — `x // w` đã đúng sẵn.

Ô dưới in bảng trên và kiểm chứng công thức "kiểu C++/Java" cho kết quả giống `//`.

In [19]:
w = 3
print(" x | x // 3 | int(x / 3)")
for x in [7, 2, 0, -1, -2, -3, -4, -7]:
    print(f"{x:>2} | {x // w:>6} | {int(x / w):>10}")


def bucket_id_c_style(x, w):
    """Mô phỏng cách viết trong C++/Java: phép chia nguyên ở đó làm tròn về 0 (giả lập bằng int(x / w))."""
    if x >= 0:
        return int(x / w)
    return int((x + 1) / w) - 1


print("Công thức C++/Java khớp // trên [-50, 50]?",
      all(bucket_id_c_style(x, w) == x // w for x in range(-50, 51)))

 x | x // 3 | int(x / 3)
 7 |      2 |          2
 2 |      0 |          0
 0 |      0 |          0
-1 |     -1 |          0
-2 |     -1 |          0
-3 |     -1 |         -1
-4 |     -2 |         -1
-7 |     -3 |         -2
Công thức C++/Java khớp // trên [-50, 50]? True


### 10.2. Ba sự thật về thùng (với `w = t + 1`)

Gọi `id(a) = a // w`. Nhớ rằng thùng `b` chứa đúng đoạn `[b·w, b·w + t]` (vì `w - 1 = t`).

**(S1) Cùng thùng ⇒ chắc chắn gần.** Hai số trong cùng đoạn `[b·w, b·w + t]` chênh nhau tối đa `(b·w + t) - b·w = t`. Ví dụ `t = 2`: thùng 1 = `{3, 4, 5}`, chênh lớn nhất là `5 - 3 = 2`.

**(S2) Cách nhau từ 2 thùng trở lên ⇒ chắc chắn xa.** Giả sử `id(b) >= id(a) + 2`. Số lớn nhất có thể của `a` là `id(a)·w + t`, số nhỏ nhất có thể của `b` là `(id(a) + 2)·w = id(a)·w + 2t + 2`. Hiệu nhỏ nhất là `t + 2 > t`. Ví dụ `t = 2`: thùng 0 = `{0, 1, 2}`, thùng 2 = `{6, 7, 8}`, gần nhất là `6 - 2 = 4 > 2`.

**(S3) Hai thùng kề nhau ⇒ chưa biết, phải so thật.** Thùng 0 và thùng 1 (`t = 2`): `2` và `3` chênh `1 <= 2` (gần), nhưng `0` và `5` chênh `5 > 2` (xa).

### Hai hệ quả làm nên thuật toán

1. **Mỗi bước chỉ cần xét 3 thùng:** thùng của `x` là `b`, và hai thùng kề `b - 1`, `b + 1`. Theo (S2), mọi thùng khác đều chứa số quá xa `x`, khỏi xét.
2. **Mỗi thùng chứa tối đa 1 phần tử của cửa sổ.** Nếu hai phần tử trong cửa sổ cùng thùng thì theo (S1) chúng gần nhau ⇒ ta đã trả `True` và dừng từ lúc phần tử thứ hai xuất hiện. Vậy chỉ cần `dict` **`buckets: id thùng → giá trị duy nhất trong thùng`**, không cần list.

Ba bước tra `dict`, mỗi bước trung bình `O(1)` ⇒ cả thuật toán `O(n)`.

**Vì sao `w = t + 1` mà không phải `t`?** Thùng rộng `t + 1` là rộng **nhất** mà (S1) vẫn đúng (rộng hơn thì hai đầu thùng chênh quá `t`). Và nếu lấy `w = t` thì khi `t = 0` sẽ chia cho 0 ⇒ `ZeroDivisionError`.

Ô dưới kiểm chứng (S1), (S2), (S3) bằng cách thử **mọi cặp** số trong `[-15, 15]` với vài giá trị `t`.

In [20]:
def check_bucket_facts(t, lo=-15, hi=15):
    w = t + 1
    s1_ok = True            # S1: cùng thùng -> luôn gần
    s2_ok = True            # S2: cách >= 2 thùng -> luôn xa
    adj_close = adj_far = 0 # S3: đếm cặp ở 2 thùng kề nhau
    for a in range(lo, hi + 1):
        for b in range(lo, hi + 1):
            gap = abs(a // w - b // w)
            close = abs(a - b) <= t
            if gap == 0 and not close:
                s1_ok = False
            if gap >= 2 and close:
                s2_ok = False
            if gap == 1:
                if close:
                    adj_close += 1
                else:
                    adj_far += 1
    print(f"t={t}: S1 đúng? {s1_ok} | S2 đúng? {s2_ok} | cặp ở thùng kề: gần={adj_close}, xa={adj_far}")


for t in [0, 1, 2, 5]:
    check_bucket_facts(t)

t=0: S1 đúng? True | S2 đúng? True | cặp ở thùng kề: gần=0, xa=60
t=1: S1 đúng? True | S2 đúng? True | cặp ở thùng kề: gần=30, xa=86
t=2: S1 đúng? True | S2 đúng? True | cặp ở thùng kề: gần=58, xa=110
t=5: S1 đúng? True | S2 đúng? True | cặp ở thùng kề: gần=142, xa=158


Nhận xét: (S1) và (S2) đúng với mọi `t` đã thử. Ở thùng kề nhau có cả cặp "gần" lẫn cặp "xa" (S3) — trừ `t = 0`, khi mọi cặp ở thùng kề đều xa (mục 10.6 sẽ giải thích).

### 10.3. Thuật toán Cách 3

```text
w = t + 1, buckets = {}                         # id thùng -> giá trị
for i, x in enumerate(nums):
    b = x // w
    (1) nếu b có trong buckets                          -> True   (S1)
    (2) nếu b-1 có trong buckets và x - buckets[b-1] <= t -> True   (S3)
    (3) nếu b+1 có trong buckets và buckets[b+1] - x <= t -> True   (S3)
    (4) buckets[b] = x                                  # thêm vào cửa sổ
    (5) nếu i >= k: del buckets[nums[i-k] // w]         # bỏ phần tử quá hạn
return False
```

**Vì sao ở (2) không cần `abs`?** Mọi số trong thùng `b - 1` đều **nhỏ hơn** mọi số trong thùng `b`, nên `x - buckets[b-1] > 0`. Tương tự ở (3), `buckets[b+1] - x > 0`. (Viết `abs(...)` cũng đúng, chỉ là không cần.)

**Vì sao bước (5) xóa đúng phần tử?** Ta xóa theo **key** `nums[i-k] // w`. Cần chắc rằng thùng đó đang chứa đúng `nums[i-k]`, không phải số khác:

- Lúc `nums[i-k]` được thêm (bước 4), thùng của nó **đang trống** — nếu không thì bước (1) đã trả `True`.
- Sau đó, không phần tử nào khác vào được thùng này khi `nums[i-k]` còn trong cửa sổ — cũng vì bước (1) sẽ trả `True` trước khi kịp ghi đè.

**Bất biến:** ở đầu bước `i`, `buckets` chứa đúng các phần tử `nums[max(0, i-k) .. i-1]`, mỗi phần tử một thùng riêng, key là `nums[j] // w`.

In [21]:
def almost_dup_bucket(nums: List[int], indexDiff: int, valueDiff: int) -> bool:
    k, t = indexDiff, valueDiff
    w = t + 1                          # độ rộng thùng (t = 0 vẫn an toàn)
    buckets = {}                       # id thùng -> giá trị duy nhất trong thùng
    for i, x in enumerate(nums):
        b = x // w                     # floor division: đúng cả với số âm
        if b in buckets:                                    # (1) cùng thùng
            return True
        if b - 1 in buckets and x - buckets[b - 1] <= t:    # (2) thùng trái
            return True
        if b + 1 in buckets and buckets[b + 1] - x <= t:    # (3) thùng phải
            return True
        buckets[b] = x                                      # (4) thêm
        if i >= k:                                          # (5) bỏ nums[i-k]
            del buckets[nums[i - k] // w]
    return False


print(almost_dup_bucket([1, 2, 3, 1], 3, 0))          # True
print(almost_dup_bucket([1, 5, 9, 1, 5, 9], 2, 3))    # False
print(almost_dup_bucket([8, -4, 3, 9, -2, 0], 2, 2))  # True
print(almost_dup_bucket([-3, -1], 1, 2))              # True

True
False
True
True


### 10.4. Trace 1 — Ví dụ 2: `nums = [1, 5, 9, 1, 5, 9]`, `k = 2`, `t = 3` ⇒ `w = 4`

Id thùng: `1 // 4 = 0`, `5 // 4 = 1`, `9 // 4 = 2`.

| `i` | `x` | `b` | `buckets` trước | Thùng `b` | Thùng `b-1` | Thùng `b+1` | Cập nhật |
|---:|---:|---:|---|---|---|---|---|
| 0 | 1 | 0 | `{}` | trống | trống | trống | thêm `0:1` ⇒ `{0:1}` |
| 1 | 5 | 1 | `{0:1}` | trống | `1`: `|5-1| = 4 > 3` | trống | thêm `1:5` ⇒ `{0:1, 1:5}` |
| 2 | 9 | 2 | `{0:1, 1:5}` | trống | `5`: `4 > 3` | trống | thêm `2:9`; xóa `nums[0]=1` (thùng 0) ⇒ `{1:5, 2:9}` |
| 3 | 1 | 0 | `{1:5, 2:9}` | **trống** | trống | `5`: `4 > 3` | thêm `0:1`; xóa `nums[1]=5` (thùng 1) ⇒ `{0:1, 2:9}` |
| 4 | 5 | 1 | `{0:1, 2:9}` | trống | `1`: `4 > 3` | `9`: `4 > 3` | thêm `1:5`; xóa `nums[2]=9` (thùng 2) ⇒ `{0:1, 1:5}` |
| 5 | 9 | 2 | `{0:1, 1:5}` | trống | `5`: `4 > 3` | trống | thêm `2:9`; xóa `nums[3]=1` (thùng 0) ⇒ `{1:5, 2:9}` |

Hết mảng ⇒ **`False`**.

Quan sát:

- Bước `i = 3`: `x = 1` thuộc thùng 0 — đúng thùng của `nums[0] = 1`. Nhưng `nums[0]` đã bị **xóa ở bước 2** vì nó cách `i = 3` tới 3 ô `> k`. Đây chính là cặp `(0, 3)` "bằng nhau nhưng quá xa" ở mục 2.
- Mọi lần so thùng kề đều hụt đúng 1 đơn vị (`4 > 3`): các giá trị `1, 5, 9` cách nhau đúng `w = 4`, nên luôn rơi vào **thùng kề** nhưng **không đủ gần**. Đây là tình huống (S3) "phải so thật".
- Hàm `trace_bucket` in `buckets` theo thứ tự key tăng dần cho dễ đọc.

In [22]:
def fmt_buckets(buckets):
    return "{" + ", ".join(f"{b}:{v}" for b, v in sorted(buckets.items())) + "}"


def fmt_diff(x, y):
    return f"|{x}-({y})|" if y < 0 else f"|{x}-{y}|"


def trace_bucket(nums, indexDiff, valueDiff):
    k, t = indexDiff, valueDiff
    w = t + 1
    buckets = {}
    print(f"nums={nums}, k={k}, t={t}, w=t+1={w}")
    for i, x in enumerate(nums):
        b = x // w
        line = f"i={i} x={x:>2} b={b:>2} | trước {fmt_buckets(buckets):<14} |"
        if b in buckets:
            print(line, f"thùng {b} đã có {buckets[b]} -> True")
            return True
        notes = []
        for nb in (b - 1, b + 1):
            if nb in buckets:
                y = buckets[nb]
                ok = abs(x - y) <= t
                notes.append(f"thùng {nb} có {y}: {fmt_diff(x, y)}={abs(x - y)}{'<=' if ok else '>'}{t}")
                if ok:
                    print(line, "; ".join(notes), "-> True")
                    return True
        print(line, "; ".join(notes) if notes else "3 thùng b-1, b, b+1 đều trống")
        buckets[b] = x
        if i >= k:
            old = nums[i - k]
            del buckets[old // w]
            print(f"      thêm {b}:{x}, xóa nums[{i - k}]={old} (thùng {old // w}) -> {fmt_buckets(buckets)}")
        else:
            print(f"      thêm {b}:{x} -> {fmt_buckets(buckets)}")
    print("Hết mảng -> False")
    return False


result = trace_bucket([1, 5, 9, 1, 5, 9], 2, 3)

nums=[1, 5, 9, 1, 5, 9], k=2, t=3, w=t+1=4
i=0 x= 1 b= 0 | trước {}             | 3 thùng b-1, b, b+1 đều trống
      thêm 0:1 -> {0:1}
i=1 x= 5 b= 1 | trước {0:1}          | thùng 0 có 1: |5-1|=4>3
      thêm 1:5 -> {0:1, 1:5}
i=2 x= 9 b= 2 | trước {0:1, 1:5}     | thùng 1 có 5: |9-5|=4>3
      thêm 2:9, xóa nums[0]=1 (thùng 0) -> {1:5, 2:9}
i=3 x= 1 b= 0 | trước {1:5, 2:9}     | thùng 1 có 5: |1-5|=4>3
      thêm 0:1, xóa nums[1]=5 (thùng 1) -> {0:1, 2:9}
i=4 x= 5 b= 1 | trước {0:1, 2:9}     | thùng 0 có 1: |5-1|=4>3; thùng 2 có 9: |5-9|=4>3
      thêm 1:5, xóa nums[2]=9 (thùng 2) -> {0:1, 1:5}
i=5 x= 9 b= 2 | trước {0:1, 1:5}     | thùng 1 có 5: |9-5|=4>3
      thêm 2:9, xóa nums[3]=1 (thùng 0) -> {1:5, 2:9}
Hết mảng -> False


### 10.5. Trace 2 — có số âm: `nums = [8, -4, 3, 9, -2, 0]`, `k = 2`, `t = 2` ⇒ `w = 3`

Id thùng (để ý số âm làm tròn **xuống**): `8 // 3 = 2`, `-4 // 3 = -2` (vì `-1.33` làm tròn xuống là `-2`), `3 // 3 = 1`, `9 // 3 = 3`, `-2 // 3 = -1`, `0 // 3 = 0`.

| `i` | `x` | `b` | `buckets` trước | Xét 3 thùng `b-1, b, b+1` | Cập nhật |
|---:|---:|---:|---|---|---|
| 0 | 8 | 2 | `{}` | đều trống | thêm `2:8` |
| 1 | -4 | -2 | `{2:8}` | thùng -3, -2, -1 đều trống | thêm `-2:-4` |
| 2 | 3 | 1 | `{-2:-4, 2:8}` | thùng 2 có `8`: `|3-8| = 5 > 2` | thêm `1:3`; xóa `nums[0]=8` (thùng 2) |
| 3 | 9 | 3 | `{-2:-4, 1:3}` | thùng 2, 3, 4 đều trống | thêm `3:9`; xóa `nums[1]=-4` (thùng -2) |
| 4 | -2 | -1 | `{1:3, 3:9}` | thùng -2, -1, 0 đều trống | thêm `-1:-2`; xóa `nums[2]=3` (thùng 1) |
| 5 | 0 | 0 | `{-1:-2, 3:9}` | thùng -1 có `-2`: `|0-(-2)| = 2 <= 2` | **True** |

Ba bước đáng học:

- **`i = 3`:** thùng 2 lẽ ra chứa `8` và `|9 - 8| = 1 <= 2`, nhưng `8` đã bị xóa ở bước 2 (nó ở chỉ số 0, cách `i = 3` tới 3 ô `> k`). Nếu quên xóa ⇒ trả `True` **sai**.
- **`i = 4`:** tương tự, thùng -2 lẽ ra chứa `-4` (`|-2 - (-4)| = 2 <= 2`), nhưng `-4` ở chỉ số 1, cách `i = 4` tới 3 ô ⇒ đã bị xóa đúng lúc.
- **`i = 5`:** tìm thấy qua **thùng kề bên trái** (-1) — cặp `(4, 5)`, khớp với brute force và Cách 2.

In [23]:
result = trace_bucket([8, -4, 3, 9, -2, 0], 2, 2)

nums=[8, -4, 3, 9, -2, 0], k=2, t=2, w=t+1=3
i=0 x= 8 b= 2 | trước {}             | 3 thùng b-1, b, b+1 đều trống
      thêm 2:8 -> {2:8}
i=1 x=-4 b=-2 | trước {2:8}          | 3 thùng b-1, b, b+1 đều trống
      thêm -2:-4 -> {-2:-4, 2:8}
i=2 x= 3 b= 1 | trước {-2:-4, 2:8}   | thùng 2 có 8: |3-8|=5>2
      thêm 1:3, xóa nums[0]=8 (thùng 2) -> {-2:-4, 1:3}
i=3 x= 9 b= 3 | trước {-2:-4, 1:3}   | 3 thùng b-1, b, b+1 đều trống
      thêm 3:9, xóa nums[1]=-4 (thùng -2) -> {1:3, 3:9}
i=4 x=-2 b=-1 | trước {1:3, 3:9}     | 3 thùng b-1, b, b+1 đều trống
      thêm -1:-2, xóa nums[2]=3 (thùng 1) -> {-1:-2, 3:9}
i=5 x= 0 b= 0 | trước {-1:-2, 3:9}   | thùng -1 có -2: |0-(-2)|=2<=2 -> True


### 10.6. Trường hợp đặc biệt: `t = 0` và `t` rất lớn

**`t = 0`** ⇒ `w = 1` ⇒ `b = x // 1 = x`: **mỗi giá trị là một thùng riêng**. Khi đó:

- (S1) "cùng thùng" nghĩa là **bằng nhau** — đúng điều #219 hỏi.
- Thùng kề `x - 1`, `x + 1` chứa số chênh `1 > 0` ⇒ không bao giờ thỏa (khớp với kết quả `gần=0` ở ô kiểm chứng mục 10.2).

Vậy với `t = 0`, Cách 3 **chính là** lời giải cửa sổ của #219 — `buckets` đóng vai trò `window`. Bài #220 đúng là "#219 mở rộng".

**`t` rất lớn** (ví dụ `t = 10^9`) ⇒ `w = 10^9 + 1`: mọi giá trị trong `[-10^9, 10^9]` chỉ rơi vào hai thùng `-1` và `0`. Thuật toán vẫn đúng: cửa sổ chỉ chứa được tối đa 2 phần tử trước khi phần tử thứ ba chắc chắn rơi vào một thùng đã có ⇒ `True`.

### 10.7. Độ phức tạp Cách 3

Gọi `n = len(nums)`, `k = indexDiff`.

- **Time: `O(n)` trung bình.** Mỗi phần tử chỉ tốn: 1 phép chia `//`, tối đa 3 lần tra `dict`, 1 lần ghi, tối đa 1 lần xóa — mỗi thao tác `dict` là `O(1)` trung bình. Không còn phụ thuộc vào `k` hay `t`!
- **Space: `O(min(n, k))`.** `buckets` chứa các phần tử của cửa sổ — tối đa `k + 1` phần tử (ngay sau bước thêm, trước bước xóa), và không quá `n`.

So với Cách 2: ta **bỏ hẳn thứ tự** (không cần sắp xếp), đổi lại chỉ trả lời được câu hỏi "có/không" — vì hệ quả "mỗi thùng tối đa 1 phần tử" dựa vào việc **dừng ngay** khi tìm thấy.

In [24]:
random.seed(2200)
for _ in range(3000):
    arr = [random.randint(-6, 6) for _ in range(random.randint(2, 12))]
    kk = random.randint(1, len(arr))
    assert almost_dup_bucket(arr, kk, 0) == contains_nearby_duplicate_219_window(arr, kk)
print("t = 0: Cách 3 khớp lời giải #219 trên 3000 test ngẫu nhiên")

w_big = 10**9 + 1
print("t = 10^9 -> id thùng của -10^9, -1, 0, 10^9:", [v // w_big for v in [-10**9, -1, 0, 10**9]])

t = 0: Cách 3 khớp lời giải #219 trên 3000 test ngẫu nhiên
t = 10^9 -> id thùng của -10^9, -1, 0, 10^9: [-1, -1, 0, 0]


## 11. Cách 4 (tham khảo) — Sắp xếp theo giá trị kèm chỉ số rồi quét

### Đổi góc nhìn

Ba cách trên đi theo **chỉ số** rồi tìm **giá trị** gần. Ta thử làm ngược lại: đi theo **giá trị** (đã sắp xếp) rồi kiểm tra **chỉ số** gần.

### Công cụ: `enumerate` và sắp xếp `tuple`

- `enumerate(nums)` cho từng cặp `(i, x)`. Ta tạo list các `tuple` `(x, i)` — giá trị đứng trước để sắp xếp theo giá trị.
- `list.sort()` trên các `tuple` sắp **theo phần tử đầu**, nếu bằng nhau thì theo phần tử thứ hai: `(3, 2) < (3, 5) < (8, 0)`.

### Thuật toán

Sau khi sắp xếp thành `pairs = [(v0, i0), (v1, i1), ...]` với `v0 <= v1 <= ...`: với mỗi vị trí `p`, quét `q = p + 1, p + 2, ...` **chừng nào** `pairs[q].v - pairs[p].v <= t`; gặp cặp nào có `abs(i_p - i_q) <= k` ⇒ `True`. Khi chênh lệch giá trị vượt `t` thì **dừng quét** cho `p` này, vì các `q` phía sau còn lớn hơn nữa.

Ví dụ `nums = [8, -4, 3, 9, -2, 0]` ⇒ `pairs = [(-4, 1), (-2, 4), (0, 5), (3, 2), (8, 0), (9, 3)]`. Với `t = 2`: tại `p = 0` (giá trị -4), quét `(-2, 4)`: chênh 2, chỉ số `|1 - 4| = 3 > 2`; tới `(0, 5)`: chênh 4 > 2 ⇒ dừng. Tại `p = 1` (giá trị -2), quét `(0, 5)`: chênh 2, chỉ số `|4 - 5| = 1 <= 2` ⇒ **True**.

In [25]:
nums = [8, -4, 3, 9, -2, 0]
pairs = []
for i, x in enumerate(nums):
    pairs.append((x, i))
pairs.sort()
print("pairs sau khi sắp xếp:", pairs)


def almost_dup_sort_scan(nums, indexDiff, valueDiff, return_count=False):
    k, t = indexDiff, valueDiff
    pairs = sorted((x, i) for i, x in enumerate(nums))   # viết gọn của 4 dòng ở trên
    comparisons = 0
    for p in range(len(pairs)):
        q = p + 1
        while q < len(pairs) and pairs[q][0] - pairs[p][0] <= t:
            comparisons += 1
            if abs(pairs[q][1] - pairs[p][1]) <= k:
                return (True, comparisons) if return_count else True
            q += 1
    return (False, comparisons) if return_count else False


print(almost_dup_sort_scan([1, 2, 3, 1], 3, 0))          # True
print(almost_dup_sort_scan([1, 5, 9, 1, 5, 9], 2, 3))    # False
print(almost_dup_sort_scan([8, -4, 3, 9, -2, 0], 2, 2))  # True

pairs sau khi sắp xếp: [(-4, 1), (-2, 4), (0, 5), (3, 2), (8, 0), (9, 3)]
True
False
True


### Hạn chế của Cách 4

**1. Không được "chỉ xét các cặp đứng kề nhau" sau khi sắp xếp.** Một cám dỗ là: "sau khi sắp xếp, cặp gần nhau nhất về giá trị luôn đứng kề nhau, vậy chỉ cần xét cặp kề". Sai! Cặp kề có thể **xa về chỉ số**, còn cặp thỏa đề lại **không kề**. Phản ví dụ: `nums = [1, 4, 100, 2]`, `k = 1`, `t = 3`:

- Đáp án `True` nhờ cặp `(0, 1)`: `|1 - 4| = 3 <= 3`, chỉ số cách 1.
- Sau sắp xếp: `(1,0), (2,3), (4,1), (100,2)`. Các cặp kề: `(1,0)-(2,3)` cách 3 chỉ số; `(2,3)-(4,1)` cách 2 chỉ số; `(4,1)-(100,2)` chênh 96 ⇒ không cặp kề nào thỏa ⇒ trả `False` **sai**. Số `2` (chỉ số 3) "chen" vào giữa `1` và `4` trong thứ tự giá trị.

**2. Trường hợp xấu nhất vẫn `O(n²)`.** Nếu có nhiều giá trị gần nhau (trong vòng `t`) nhưng **chỉ số xa nhau**, vòng `while` không dừng sớm được. Ví dụ `k = 1`, `t = m` và mảng xen kẽ `[0, 10^6, 1, 10^6+1, 2, 10^6+2, ...]` (`n = 2m`): các số nhỏ `0..m-1` đều cách nhau `<= t` nhưng luôn đứng cách nhau ít nhất 2 chỉ số ⇒ mỗi số phải quét gần hết nhóm của nó ⇒ khoảng `m²` phép so sánh `≈ n²/4`.

**3.** Phải sắp xếp cả mảng (`O(n log n)`, tốn `O(n)` bộ nhớ) và không xử lý được kiểu "dữ liệu đến dần" như cửa sổ trượt.

⇒ Cách 4 là một góc nhìn hay để luyện tư duy, nhưng **không** dùng để nộp. Ô dưới minh họa hạn chế 1 và 2: khi `n` gấp đôi, số phép so sánh gấp khoảng **4 lần** — dấu hiệu của `O(n²)`.

In [26]:
def almost_dup_adjacent_only_WRONG(nums, indexDiff, valueDiff):
    pairs = sorted((x, i) for i, x in enumerate(nums))
    for p in range(len(pairs) - 1):
        (v1, i1), (v2, i2) = pairs[p], pairs[p + 1]
        if v2 - v1 <= valueDiff and abs(i1 - i2) <= indexDiff:
            return True
    return False


print("Chỉ xét cặp kề :", almost_dup_adjacent_only_WRONG([1, 4, 100, 2], 1, 3), "(sai)")
print("Brute force    :", almost_dup_bruteforce([1, 4, 100, 2], 1, 3))


def worst_case_for_sort_scan(m):
    arr = []
    for v in range(m):
        arr.append(v)              # nhóm nhỏ ở chỉ số chẵn
        arr.append(10**6 + v)      # nhóm lớn ở chỉ số lẻ
    return arr


for m in [250, 500, 1000]:
    arr = worst_case_for_sort_scan(m)
    ans, cmp_count = almost_dup_sort_scan(arr, 1, m, return_count=True)
    print(f"n={len(arr):>4}: kết quả={ans}, số phép so sánh={cmp_count:>7}")

Chỉ xét cặp kề : False (sai)
Brute force    : True
n= 500: kết quả=False, số phép so sánh=  62250
n=1000: kết quả=False, số phép so sánh= 249500


n=2000: kết quả=False, số phép so sánh= 999000


## 12. Code nộp LeetCode

Ta nộp **Cách 3 (bucket)**: `O(n)` thời gian, `O(min(n, k))` bộ nhớ, chỉ dùng `dict`.

- Giữ **đúng tên** method `containsNearbyAlmostDuplicate` và tên tham số `indexDiff`, `valueDiff`.
- Code submit gọn, **không `print`**.
- Trên LeetCode, `List` đã được import sẵn; trong notebook ta đã `from typing import List` ở đầu.

Ô kế tiếp chạy test với ví dụ của đề và các edge case: số âm, `t = 0`, `k = n`, số trùng, giá trị biên `±10^9`, `t = 10^9`, cặp nằm **đúng** ở khoảng cách `k`. Sau đó là **test đối chiếu chéo**: chạy cả 4 cách trên hàng nghìn input ngẫu nhiên nhỏ (cố định `random.seed` để chạy lại ra cùng kết quả) và so với brute force.

In [27]:
class Solution:
    def containsNearbyAlmostDuplicate(self, nums: List[int], indexDiff: int, valueDiff: int) -> bool:
        w = valueDiff + 1
        buckets = {}
        for i, x in enumerate(nums):
            b = x // w
            if b in buckets:
                return True
            if b - 1 in buckets and x - buckets[b - 1] <= valueDiff:
                return True
            if b + 1 in buckets and buckets[b + 1] - x <= valueDiff:
                return True
            buckets[b] = x
            if i >= indexDiff:
                del buckets[nums[i - indexDiff] // w]
        return False


sol = Solution()
tests = [
    # (nums, indexDiff, valueDiff, expected, mô tả)
    ([1, 2, 3, 1], 3, 0, True, "Ví dụ 1"),
    ([1, 5, 9, 1, 5, 9], 2, 3, False, "Ví dụ 2"),
    ([1, 2, 3, 1], 2, 0, False, "cặp bằng nhau nhưng cách 3 > k = 2"),
    ([1, 0, 1, 1], 1, 2, True, "cặp kề nhau"),
    ([-3, -1], 1, 2, True, "số âm, chênh đúng t"),
    ([-2, 2], 1, 2, False, "số âm/dương chênh 4 > t (bẫy int(x/w))"),
    ([-1, -1], 1, 0, True, "hai số âm bằng nhau, t = 0"),
    ([8, -4, 3, 9, -2, 0], 2, 2, True, "ví dụ xuyên suốt"),
    ([10, 100, 1000, 10000], 4, 5, False, "k = n nhưng giá trị xa"),
    ([4, 1, 6, 3], 3, 0, False, "t = 0, không có số trùng"),
    ([1, 4, 7, 2], 1, 2, False, "cặp (0,3) gần giá trị nhưng xa chỉ số"),
    ([5, 0, 20, 9], 3, 4, True, "cặp duy nhất (0,3) nằm đúng ở khoảng cách k = 3"),
    ([5, 0, 20, 9], 2, 4, False, "cùng mảng nhưng k = 2 thì cặp (0,3) quá xa"),
    ([-10**9, 10**9], 1, 10**9, False, "biên: chênh 2·10^9 > 10^9"),
    ([-10**9, 0], 1, 10**9, True, "biên: chênh đúng 10^9"),
    ([10**9, 10**9 - 7], 1, 10**9, True, "t = 10^9"),
]
for nums_, k_, t_, expected, desc in tests:
    got = sol.containsNearbyAlmostDuplicate(nums_, k_, t_)
    assert got == expected, (desc, nums_, k_, t_, got, expected)
    print(f"PASS: {desc}")

PASS: Ví dụ 1
PASS: Ví dụ 2
PASS: cặp bằng nhau nhưng cách 3 > k = 2
PASS: cặp kề nhau
PASS: số âm, chênh đúng t
PASS: số âm/dương chênh 4 > t (bẫy int(x/w))
PASS: hai số âm bằng nhau, t = 0
PASS: ví dụ xuyên suốt
PASS: k = n nhưng giá trị xa
PASS: t = 0, không có số trùng
PASS: cặp (0,3) gần giá trị nhưng xa chỉ số
PASS: cặp duy nhất (0,3) nằm đúng ở khoảng cách k = 3
PASS: cùng mảng nhưng k = 2 thì cặp (0,3) quá xa
PASS: biên: chênh 2·10^9 > 10^9
PASS: biên: chênh đúng 10^9
PASS: t = 10^9


In [28]:
random.seed(220)
methods = [almost_dup_bisect, almost_dup_bucket, almost_dup_sort_scan,
           Solution().containsNearbyAlmostDuplicate]
count_true = count_false = 0
for _ in range(5000):
    n = random.randint(2, 12)
    arr = [random.randint(-20, 20) for _ in range(n)]
    k = random.randint(1, n)
    t = random.randint(0, 10)
    expected = almost_dup_bruteforce(arr, k, t)
    for f in methods:
        assert f(arr, k, t) == expected, (f.__name__, arr, k, t)
    if expected:
        count_true += 1
    else:
        count_false += 1
print(f"4 cách khớp brute force trên 5000 test ngẫu nhiên ({count_true} True, {count_false} False)")

4 cách khớp brute force trên 5000 test ngẫu nhiên (3935 True, 1065 False)


## 13. Vì sao Cách 3 đúng?

**Bất biến (invariant):** ở đầu mỗi bước `i`,

1. `buckets` chứa đúng các phần tử của cửa sổ `nums[max(0, i-k) .. i-1]`, mỗi phần tử nằm ở key `nums[j] // w` của nó;
2. mỗi key chứa đúng **một** phần tử (không có hai phần tử của cửa sổ cùng thùng);
3. chưa có cặp nào trong các chỉ số `< i` thỏa đề (nếu có, ta đã trả `True`).

Bất biến đúng lúc đầu (`buckets` rỗng). Mục 10.3 đã giải thích vì sao các bước (4), (5) giữ nó đúng cho bước `i + 1`.

**Không trả `True` sai.** Mỗi lần trả `True`, ta đã tìm thấy một phần tử `y = nums[j]` trong cửa sổ — tức `j < i` (nên `j != i`) và `i - j <= k` — với `|x - y| <= t` (ở (1) nhờ (S1), ở (2), (3) nhờ so trực tiếp). Đó là một cặp thật.

**Không bỏ sót.** Giả sử có cặp thỏa đề; xét cặp có chỉ số lớn hơn là `i` **nhỏ nhất**, với chỉ số nhỏ hơn `j < i`, `i - j <= k`, `|nums[i] - nums[j]| <= t`. Đến bước `i`, thuật toán chưa dừng (theo bất biến 3), và `nums[j]` đang nằm trong `buckets` (theo bất biến 1). Theo (S2), thùng của `nums[j]` chỉ có thể là `b - 1`, `b` hoặc `b + 1` — đúng ba thùng ta xét — nên ta phát hiện được nó (hoặc một cặp khác) và trả `True`.

**Vì sao chỉ cần nhìn về bên trái?** Mỗi cặp `(j, i)` với `j < i` được xét đúng lúc ta đứng ở chỉ số **lớn hơn** `i`: khi đó `nums[j]` đã nằm sẵn trong cửa sổ. Nên không cần nhìn sang phải.

Cách 2 đúng theo cùng lập luận, chỉ khác là "tìm phần tử gần" bằng ceiling `c` (mục 7) thay cho ba thùng.

## 14. Độ phức tạp và so sánh các cách

Ký hiệu: `n = len(nums)`, `k = indexDiff`. "Trung bình" ở Cách 3 là vì thao tác `dict` là `O(1)` **trung bình** (hashing), không phải tuyệt đối trong mọi tình huống.

| Cách | Ý tưởng | Time | Space | Nhận xét |
|---|---|---|---|---|
| 1. Brute force | so `x` với từng phần tử trong cửa sổ | `O(n·k)`, xấu nhất `O(n²)` | `O(1)` | dễ nghĩ, TLE khi `n = k = 10^5` |
| 2. List sắp xếp + `bisect` | giữ cửa sổ sắp xếp, hỏi ceiling của `x - t` | tìm `O(log k)`, chèn/xóa `list` `O(k)` ⇒ `O(n·k)` lý thuyết; `O(n log k)` với cây cân bằng / `SortedList` | `O(min(n, k))` | dịch phần tử bằng C nên thực tế chạy nhanh; ý tưởng dùng được ở rất nhiều bài |
| 3. Bucket (**chính**) | thùng rộng `t + 1`, chỉ xét `b - 1, b, b + 1` | `O(n)` trung bình | `O(min(n, k))` | tối ưu, ngắn, nhưng chỉ trả lời được câu hỏi **có/không** |
| 4. Sắp xếp + quét | sắp `(giá trị, chỉ số)`, quét khi chênh `<= t` | `O(n log n)` + quét xấu nhất `O(n²)` | `O(n)` | góc nhìn khác; không nên nộp |

Ô dưới đo thời gian thật trên input **khó nhất** (đáp án `False`, phải đi hết mảng): các giá trị khác nhau đôi một, xáo trộn ngẫu nhiên, `t = 0`, `k = n`. Con số cụ thể tùy máy, nhưng thứ tự nhanh/chậm thì ổn định.

In [29]:
def timeit(f, *args):
    start = time.perf_counter()
    result = f(*args)
    return result, time.perf_counter() - start


random.seed(14)
small = random.sample(range(-10**9, 10**9), 3000)
print("n = k = 3000:")
for f in [almost_dup_bruteforce, almost_dup_bisect, almost_dup_bucket, almost_dup_sort_scan]:
    res, sec = timeit(f, small, 3000, 0)
    print(f"  {f.__name__:<22} -> {res}, {sec:.4f} giây")

big = random.sample(range(-10**9, 10**9), 10**5)
print("n = k = 10^5 (giới hạn của đề; bỏ qua brute force):")
for f in [almost_dup_bisect, almost_dup_bucket]:
    res, sec = timeit(f, big, 10**5, 0)
    print(f"  {f.__name__:<22} -> {res}, {sec:.4f} giây")

n = k = 3000:


  almost_dup_bruteforce  -> False, 0.3150 giây
  almost_dup_bisect      -> False, 0.0018 giây
  almost_dup_bucket      -> False, 0.0007 giây
  almost_dup_sort_scan   -> False, 0.0011 giây
n = k = 10^5 (giới hạn của đề; bỏ qua brute force):


  almost_dup_bisect      -> False, 0.7844 giây
  almost_dup_bucket      -> False, 0.0522 giây


## 15. Lỗi thường gặp

### Lỗi 1 — Tính id thùng bằng `int(x / w)` (làm tròn về 0)

```python
b = int(x / w)      # SAI với số âm
```

**Vì sao sai:** `int(-2 / 3) = 0` và `int(2 / 3) = 0` ⇒ `-2` và `2` "cùng thùng 0", nhưng chênh nhau `4 > t = 2`. Bước (1) "cùng thùng ⇒ True" trả lời **sai**. Gốc rễ: thùng 0 lúc này rộng 5 giá trị (`-2..2`), phá vỡ (S1). (Ngoài ra `x / w` là phép chia số thực, có thể sai số với số lớn.)

**Sửa:** dùng `b = x // w` (Python). Trong C++/Java dùng công thức `x >= 0 ? x / w : (x + 1) / w - 1` hoặc `Math.floorDiv`.

### Lỗi 2 — Lấy độ rộng thùng `w = t`

```python
w = valueDiff       # SAI: t = 0 -> chia cho 0
```

**Vì sao sai:** đề cho phép `valueDiff = 0` ⇒ `x // 0` gây `ZeroDivisionError`. **Sửa:** `w = valueDiff + 1` — vừa an toàn, vừa là độ rộng lớn nhất để (S1) đúng.

In [30]:
def almost_dup_bucket_WRONG_trunc(nums, indexDiff, valueDiff):
    k, t = indexDiff, valueDiff
    w = t + 1
    buckets = {}
    for i, x in enumerate(nums):
        b = int(x / w)                         # SAI: làm tròn về 0
        if b in buckets:
            return True
        if b - 1 in buckets and abs(x - buckets[b - 1]) <= t:
            return True
        if b + 1 in buckets and abs(buckets[b + 1] - x) <= t:
            return True
        buckets[b] = x
        if i >= k:
            del buckets[int(nums[i - k] / w)]
    return False


print("int(x / w):", almost_dup_bucket_WRONG_trunc([-2, 2], 1, 2), "(sai, |-2-2| = 4 > 2)")
print("x // w    :", almost_dup_bucket([-2, 2], 1, 2), "(đúng)")

try:
    w = 0                                      # w = t với t = 0
    print(5 // w)
except ZeroDivisionError as e:
    print("w = t khi t = 0 -> ZeroDivisionError:", e)

int(x / w): True (sai, |-2-2| = 4 > 2)
x // w    : False (đúng)
w = t khi t = 0 -> ZeroDivisionError: integer division or modulo by zero


### Lỗi 3 — Xóa sai thời điểm / sai chỉ số (off-by-one)

```python
for i, x in enumerate(nums):
    if i >= k:
        del buckets[nums[i - k] // w]   # SAI: xóa TRƯỚC khi kiểm tra
    ...
```

**Vì sao sai:** ở đầu bước `i`, cửa sổ đúng phải là `nums[i-k .. i-1]` — **có** `nums[i - k]`, vì `i - (i - k) = k <= k` vẫn hợp lệ. Xóa nó trước khi kiểm tra là bỏ mất các cặp cách **đúng `k`**. Ví dụ 1 (`(0, 3)` cách đúng `k = 3`) sẽ trả `False` sai.

**Sửa:** kiểm tra → thêm → **rồi mới** xóa `nums[i - k]` (hoặc, nếu muốn xóa ở đầu vòng, xóa `nums[i - k - 1]` khi `i > k`).

### Lỗi 4 — Thêm `x` trước khi kiểm tra

**Vì sao sai:** `x` sẽ so với **chính nó** (cùng thùng, `|x - x| = 0`) ⇒ luôn `True`, vi phạm `i != j`. Đây đúng là bài học "kiểm tra trước, thêm sau" từ #217.

In [31]:
def almost_dup_bucket_WRONG_remove_early(nums, indexDiff, valueDiff):
    k, t = indexDiff, valueDiff
    w = t + 1
    buckets = {}
    for i, x in enumerate(nums):
        if i >= k:
            del buckets[nums[i - k] // w]         # SAI: xóa TRƯỚC khi kiểm tra
        b = x // w
        if b in buckets:
            return True
        if b - 1 in buckets and x - buckets[b - 1] <= t:
            return True
        if b + 1 in buckets and buckets[b + 1] - x <= t:
            return True
        buckets[b] = x
    return False


def almost_dup_bucket_WRONG_add_first(nums, indexDiff, valueDiff):
    k, t = indexDiff, valueDiff
    w = t + 1
    buckets = {}
    for i, x in enumerate(nums):
        b = x // w
        buckets[b] = x                            # SAI: thêm trước
        if b in buckets:                          # luôn đúng -> tự so với chính mình
            return True
        if i >= k:
            del buckets[nums[i - k] // w]
    return False


print("Ví dụ 1 (đúng là True):")
print("  xóa trước khi kiểm tra:", almost_dup_bucket_WRONG_remove_early([1, 2, 3, 1], 3, 0))
print("  bản đúng              :", almost_dup_bucket([1, 2, 3, 1], 3, 0))
print("Ví dụ 2 (đúng là False):")
print("  thêm trước khi kiểm tra:", almost_dup_bucket_WRONG_add_first([1, 5, 9, 1, 5, 9], 2, 3))
print("  bản đúng               :", almost_dup_bucket([1, 5, 9, 1, 5, 9], 2, 3))

Ví dụ 1 (đúng là True):
  xóa trước khi kiểm tra: False
  bản đúng              : True
Ví dụ 2 (đúng là False):
  thêm trước khi kiểm tra: True
  bản đúng               : False


### Lỗi 5 — Cách 2: tìm `bisect_left(window, x)` rồi chỉ xem phần tử bên phải

```python
pos = bisect.bisect_left(window, x)           # SAI: chỉ tìm số >= x
if pos < len(window) and window[pos] - x <= t:
    return True
```

**Vì sao sai:** cách này chỉ nhìn **số liền sau** `x` (successor) và bỏ quên **số liền trước** `window[pos - 1]` (predecessor), trong khi số gần `x` có thể nhỏ hơn `x`. Ví dụ `nums = [1, 3]`, `k = 1`, `t = 2`: khi `x = 3`, `window = [1]`, `bisect_left([1], 3) = 1 = len` ⇒ trả `False`, nhưng `|3 - 1| = 2 <= 2` ⇒ đúng ra là `True`.

**Sửa:** tìm ceiling của **`x - t`** (mục 7) — một lần tìm là đủ. (Hoặc kiểm tra cả `window[pos - 1]` và `window[pos]`.)

### Các lỗi "lặt vặt" khác

- **Dùng `<` thay cho `<=`:** đề ghi `abs(i - j) <= indexDiff` và `abs(...) <= valueDiff`. Cặp chênh **đúng** `t` hoặc cách **đúng** `k` vẫn hợp lệ (test `[-3, -1]` và `[5, 0, 20, 9]` ở mục 12 bắt lỗi này).
- **Tràn số ở C++/Java:** với constraints hiện tại, `x - t` và `x + t` nằm trong `[-2·10^9, 2·10^9]` — vẫn vừa `int` 32-bit (tối đa `2^31 - 1 ≈ 2.147·10^9`) nhưng **rất sát nút**; bản cũ của đề cho phạm vi giá trị rộng hơn (cỡ `2^31 - 1`) nên rất nhiều lời giải từng bị tràn số. Thói quen an toàn: dùng `long`. Python không bị tràn số nguyên.
- **Cách 4 chỉ xét cặp kề sau khi sắp xếp:** sai như phản ví dụ ở mục 11.
- **Quên rằng `indexDiff` có thể bằng `n`:** khi đó `i >= k` không bao giờ đúng ⇒ không xóa gì — vẫn đúng, không cần xử lý riêng.

In [32]:
def almost_dup_bisect_WRONG_successor_only(nums, indexDiff, valueDiff):
    k, t = indexDiff, valueDiff
    window = []
    for i, x in enumerate(nums):
        pos = bisect.bisect_left(window, x)          # SAI: chỉ nhìn số >= x
        if pos < len(window) and window[pos] - x <= t:
            return True
        bisect.insort(window, x)
        if i >= k:
            window.pop(bisect.bisect_left(window, nums[i - k]))
    return False


print("Chỉ nhìn số >= x :", almost_dup_bisect_WRONG_successor_only([1, 3], 1, 2), "(sai)")
print("Ceiling của x - t:", almost_dup_bisect([1, 3], 1, 2), "(đúng)")

Chỉ nhìn số >= x : False (sai)
Ceiling của x - t: True (đúng)


## 16. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu → kỹ thuật

| Đề nói... | Nghĩ ngay tới |
|---|---|
| "hai phần tử cách nhau không quá `k` vị trí" | **cửa sổ trượt** cỡ `k`: thêm `nums[i]`, bỏ `nums[i - k]` |
| "giá trị **bằng nhau**" | `set` / `dict` (#217, #219) |
| "giá trị chênh không quá `t`" | (a) cấu trúc **có thứ tự**: list sắp xếp + `bisect`, `SortedList`, `TreeSet` — hỏi **ceiling của `x - t`**; (b) **bucket** rộng `t + 1` |
| "khoảng cách giữa các giá trị", "nguyên lý chuồng bồ câu" | **bucket** (ví dụ #164) |
| cần **đếm** hoặc **tìm min/max** chênh lệch, không chỉ có/không | list sắp xếp + `bisect` (bucket khó dùng vì nó dựa vào việc dừng sớm) |

**Câu chốt dễ nhớ:** *bucket = băm theo **khoảng** thay vì băm theo **giá trị**. Thùng rộng `t + 1` ⇒ cùng thùng chắc chắn gần, cách 2 thùng chắc chắn xa, chỉ thùng kề mới phải so.*

### Bài liên quan

- **#217 Contains Duplicate** và **#219 Contains Duplicate II** — đã ôn ở mục 3, 4. `#220` với `t = 0` chính là `#219`.
- **#164 Maximum Gap** (Hard) — dùng bucket + nguyên lý chuồng bồ câu để tìm khoảng cách lớn nhất giữa hai số liên tiếp sau sắp xếp trong `O(n)`.
- **#2817 Minimum Absolute Difference Between Elements With Constraint** — list sắp xếp + `bisect`, xét cả số liền trước và liền sau (Bài 6 bên dưới).
- **#1438 Longest Continuous Subarray With Absolute Diff Less Than or Equal to Limit** — cửa sổ trượt kết hợp cấu trúc biết min/max (list sắp xếp hoặc monotonic deque).
- **Biến thể tự chế:** trả về **chính cặp chỉ số** (Bài 4), **đếm** số cặp thỏa (Bài 5).

## 17. Bài tập tự luyện

Làm theo thứ tự, mỗi bài khó hơn bài trước. Gợi ý chia 3 mức — chỉ mở mức tiếp theo khi bí. **Đáp án ở cuối notebook.**

### Bài 1 — Đoán output (không chạy code)

1. Với `t = 4` (tức `w = 5`), tính `x // 5` cho `x` lần lượt là `14, 5, 4, 0, -1, -5, -6, -11`.
2. `-7 // 2` bằng bao nhiêu? `int(-7 / 2)` bằng bao nhiêu?
3. `almost_dup_bucket([-1, 4], 1, 4)` trả về gì? Hai số này ở thùng nào?

Gợi ý mức 1: `//` luôn làm tròn **xuống**, kể cả với số âm.
Gợi ý mức 2: thùng `b` chứa đoạn `[5b, 5b + 4]`; ví dụ thùng -1 là `[-5, -1]`.
Gợi ý mức 3: câu 3 — hai thùng kề nhau, nên phải so thật `|4 - (-1)|` với `4`.

### Bài 2 — Tính tay Cách 3

Lập bảng trace Cách 3 (giống mục 10.4) cho `nums = [4, 11, -3, 6, 0]`, `k = 2`, `t = 3`. Cột: `i`, `x`, `b`, `buckets` trước, 3 thùng được xét, cập nhật. Câu hỏi phụ: ở bước `i = 3` (`x = 6`), vì sao **không** tính cặp với số `4` dù `|6 - 4| = 2 <= 3`?

Gợi ý mức 1: `w = 4`; tính trước id thùng của cả 5 số.
Gợi ý mức 2: id thùng lần lượt là `1, 2, -1, 1, 0`.
Gợi ý mức 3: số `4` ở chỉ số 0; tới bước `i = 2` nó đã bị xóa vì `i >= k`.

### Bài 3 — Điền TODO: Cách 2 (list sắp xếp + `bisect`)

Hoàn thiện hàm theo 4 TODO. Đừng nhìn lại mục 9!

Gợi ý mức 1: ba việc mỗi bước theo đúng thứ tự: **hỏi → thêm → bỏ**.
Gợi ý mức 2: hỏi bằng `bisect.bisect_left(window, x - valueDiff)`; thêm bằng `bisect.insort`; bỏ bằng `window.pop(bisect.bisect_left(window, giá_trị_cần_bỏ))`.
Gợi ý mức 3: điều kiện trả `True` là `pos < len(window) and window[pos] <= x + valueDiff`.

In [33]:
def exercise_almost_dup_bisect(nums, indexDiff, valueDiff):
    window = []                          # luôn là list ĐÃ SẮP XẾP các giá trị trong cửa sổ
    for i, x in enumerate(nums):
        # TODO 1: pos = vị trí số nhỏ nhất trong window mà >= x - valueDiff
        # TODO 2: nếu pos hợp lệ và window[pos] <= x + valueDiff -> return True
        # TODO 3: chèn x vào window mà vẫn giữ thứ tự
        # TODO 4: nếu i >= indexDiff, xóa một bản sao của nums[i - indexDiff] khỏi window
        pass
    return False

# Tự làm xong rồi mới mở test:
# assert exercise_almost_dup_bisect([1, 2, 3, 1], 3, 0) == True
# assert exercise_almost_dup_bisect([1, 5, 9, 1, 5, 9], 2, 3) == False
# assert exercise_almost_dup_bisect([8, -4, 3, 9, -2, 0], 2, 2) == True
# assert exercise_almost_dup_bisect([1, 3], 1, 2) == True

### Bài 4 — Tự viết lại: Cách 3 trả về **chính cặp chỉ số**

Viết `find_almost_dup_pair(nums, indexDiff, valueDiff)` theo Cách 3 nhưng trả về `tuple` `(j, i)` với `j < i` là **cặp đầu tiên** thuật toán phát hiện (tức `i` nhỏ nhất có thể), hoặc `None` nếu không có.

Ví dụ: `[1, 2, 3, 1], 3, 0 → (0, 3)`; `[8, -4, 3, 9, -2, 0], 2, 2 → (4, 5)`; Ví dụ 2 `→ None`.

Gợi ý mức 1: `buckets` cần nhớ thêm **chỉ số** của phần tử trong thùng.
Gợi ý mức 2: lưu `buckets[b] = (x, i)`; khi đọc thì `y, j = buckets[b - 1]`.
Gợi ý mức 3: bước xóa không đổi — key vẫn là `nums[i - k] // w`.

In [34]:
def find_almost_dup_pair(nums, indexDiff, valueDiff):
    # TODO 1: w = valueDiff + 1, buckets = {}   (id thùng -> (giá trị, chỉ số))
    # TODO 2: với mỗi i, x: tính b; xét thùng b, b-1, b+1; nếu gặp thì return (j, i)
    # TODO 3: thêm (x, i) vào thùng b; nếu i >= indexDiff thì xóa thùng của nums[i - indexDiff]
    # TODO 4: return None
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert find_almost_dup_pair([1, 2, 3, 1], 3, 0) == (0, 3)
# assert find_almost_dup_pair([8, -4, 3, 9, -2, 0], 2, 2) == (4, 5)
# assert find_almost_dup_pair([1, 5, 9, 1, 5, 9], 2, 3) is None

### Bài 5 — Biến thể: **đếm** số cặp thỏa đề

Viết `count_almost_dup_pairs(nums, indexDiff, valueDiff)` đếm số cặp `(j, i)`, `j < i`, `i - j <= indexDiff`, `abs(nums[i] - nums[j]) <= valueDiff`. Phải đạt tốt hơn brute force: dùng cửa sổ sắp xếp + `bisect`.

Ví dụ: `[1, 2, 3, 1], 3, 1 → 4` (các cặp `(0,1), (0,3), (1,2), (1,3)`); Ví dụ 2 `→ 0`.

Câu hỏi suy nghĩ: vì sao Cách 3 (bucket) **không** dùng trực tiếp được cho bài đếm?

Gợi ý mức 1: số phần tử của `window` nằm trong `[x - t, x + t]` = (vị trí đầu tiên `> x + t`) − (vị trí đầu tiên `>= x - t`).
Gợi ý mức 2: `bisect_right(window, x + t) - bisect_left(window, x - t)`.
Gợi ý mức 3: cộng số đó vào `total` **trước** khi `insort(window, x)`; không `return` sớm.

In [35]:
def count_almost_dup_pairs(nums, indexDiff, valueDiff):
    # TODO 1: window = [], total = 0
    # TODO 2: với mỗi i, x: total += số phần tử của window trong [x - valueDiff, x + valueDiff]
    # TODO 3: insort x; nếu i >= indexDiff thì xóa nums[i - indexDiff]
    # TODO 4: return total
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert count_almost_dup_pairs([1, 2, 3, 1], 3, 1) == 4
# assert count_almost_dup_pairs([1, 5, 9, 1, 5, 9], 2, 3) == 0
# assert count_almost_dup_pairs([7, 7, 7], 2, 0) == 3

### Bài 6 — Mini challenge: LeetCode 2817 (Minimum Absolute Difference Between Elements With Constraint)

Đề (tóm tắt): cho mảng `nums` (chỉ số từ 0) và số nguyên `x`. Tìm **giá trị nhỏ nhất** của `abs(nums[i] - nums[j])` trong các cặp có `abs(i - j) >= x` (cách nhau **ít nhất** `x` vị trí). Constraints: `1 <= nums.length <= 10^5`, `1 <= nums[i] <= 10^9`, `0 <= x < nums.length`.

Ví dụ: `nums = [4, 3, 2, 4], x = 2 → 0`; `nums = [5, 3, 2, 10, 15], x = 1 → 1`; `nums = [1, 2, 3, 4], x = 3 → 3`.

Lưu ý điều kiện bây giờ **ngược** với #220: chỉ số phải **xa** (`>= x`), không phải gần. Cửa sổ trở thành "mọi phần tử đứng trước `i` ít nhất `x` ô" — nó chỉ **lớn dần**, không bao giờ phải xóa.

Gợi ý mức 1: duyệt `i` từ `x` đến `n - 1`; trước khi xét `nums[i]`, thêm `nums[i - x]` vào list sắp xếp.
Gợi ý mức 2: số gần `nums[i]` nhất trong list là số liền sau (`sorted_list[pos]`) hoặc liền trước (`sorted_list[pos - 1]`), với `pos = bisect_left(sorted_list, nums[i])`.
Gợi ý mức 3: nhớ kiểm tra `pos < len(...)` và `pos > 0` trước khi đọc; cập nhật `best = min(best, ...)`.

In [36]:
def min_abs_diff_with_gap(nums, x):
    # TODO 1: sorted_list = [], best = vô cùng (float("inf"))
    # TODO 2: for i in range(x, len(nums)): insort nums[i - x]
    # TODO 3: pos = bisect_left(sorted_list, nums[i]); xét sorted_list[pos] và sorted_list[pos - 1]
    # TODO 4: return best
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert min_abs_diff_with_gap([4, 3, 2, 4], 2) == 0
# assert min_abs_diff_with_gap([5, 3, 2, 10, 15], 1) == 1
# assert min_abs_diff_with_gap([1, 2, 3, 4], 3) == 3

## 18. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

1. `w = 5`: `14 // 5 = 2`, `5 // 5 = 1`, `4 // 5 = 0`, `0 // 5 = 0`, `-1 // 5 = -1`, `-5 // 5 = -1`, `-6 // 5 = -2`, `-11 // 5 = -3`. (Thùng -1 là `[-5, -1]`, thùng -2 là `[-10, -6]`, thùng -3 là `[-15, -11]`.)
2. `-7 // 2 = -4` (vì `-3.5` làm tròn **xuống** là `-4`); `int(-7 / 2) = -3` (làm tròn **về 0**).
3. `-1` ở thùng `-1`, `4` ở thùng `0` ⇒ thùng kề ⇒ so thật: `|4 - (-1)| = 5 > 4` ⇒ **`False`**.

### Đáp án Bài 2

`w = 4`; id thùng: `4 → 1`, `11 → 2`, `-3 → -1`, `6 → 1`, `0 → 0`.

| `i` | `x` | `b` | `buckets` trước | Xét 3 thùng | Cập nhật |
|---:|---:|---:|---|---|---|
| 0 | 4 | 1 | `{}` | đều trống | thêm `1:4` |
| 1 | 11 | 2 | `{1:4}` | thùng 1 có `4`: `|11-4| = 7 > 3` | thêm `2:11` |
| 2 | -3 | -1 | `{1:4, 2:11}` | thùng -2, -1, 0 đều trống | thêm `-1:-3`; xóa `nums[0]=4` (thùng 1) |
| 3 | 6 | 1 | `{-1:-3, 2:11}` | thùng 1 **trống** (4 đã bị xóa); thùng 2 có `11`: `|6-11| = 5 > 3` | thêm `1:6`; xóa `nums[1]=11` (thùng 2) |
| 4 | 0 | 0 | `{-1:-3, 1:6}` | thùng -1 có `-3`: `|0-(-3)| = 3 <= 3` | **True** — cặp `(2, 4)` |

Câu hỏi phụ: số `4` ở chỉ số 0, còn `x = 6` ở chỉ số 3 ⇒ cách nhau `3 > k = 2` ⇒ không hợp lệ; thuật toán đã xóa `4` khỏi `buckets` ở cuối bước `i = 2`.

Ô code dưới in lại trace Bài 2 bằng `trace_bucket` để đối chiếu, rồi chạy đáp án Bài 3 → 6 kèm assert và test chéo với brute force.

**Bài 5 — vì sao bucket không đếm được?** Bucket dựa vào hệ quả "mỗi thùng tối đa 1 phần tử", mà hệ quả này chỉ đúng vì ta **dừng ngay** khi gặp cặp thỏa. Bài đếm không được dừng, nên một thùng có thể chứa nhiều phần tử, và ở thùng kề ta lại phải đếm xem bao nhiêu phần tử đủ gần — mất đi sự đơn giản `O(1)`.

In [37]:
# ---------- Bài 1 & Bài 2: kiểm chứng ----------
print([v // 5 for v in [14, 5, 4, 0, -1, -5, -6, -11]], -7 // 2, int(-7 / 2), almost_dup_bucket([-1, 4], 1, 4))
assert trace_bucket([4, 11, -3, 6, 0], 2, 3) == True
print()


# ---------- Bài 3 ----------
def answer_almost_dup_bisect(nums, indexDiff, valueDiff):
    window = []
    for i, x in enumerate(nums):
        pos = bisect.bisect_left(window, x - valueDiff)
        if pos < len(window) and window[pos] <= x + valueDiff:
            return True
        bisect.insort(window, x)
        if i >= indexDiff:
            window.pop(bisect.bisect_left(window, nums[i - indexDiff]))
    return False


# ---------- Bài 4 ----------
def answer_find_almost_dup_pair(nums, indexDiff, valueDiff):
    w = valueDiff + 1
    buckets = {}                                   # id thùng -> (giá trị, chỉ số)
    for i, x in enumerate(nums):
        b = x // w
        for nb in (b, b - 1, b + 1):
            if nb in buckets:
                y, j = buckets[nb]
                if abs(x - y) <= valueDiff:
                    return (j, i)
        buckets[b] = (x, i)
        if i >= indexDiff:
            del buckets[nums[i - indexDiff] // w]
    return None


# ---------- Bài 5 ----------
def answer_count_almost_dup_pairs(nums, indexDiff, valueDiff):
    window, total = [], 0
    for i, x in enumerate(nums):
        total += bisect.bisect_right(window, x + valueDiff) - bisect.bisect_left(window, x - valueDiff)
        bisect.insort(window, x)
        if i >= indexDiff:
            window.pop(bisect.bisect_left(window, nums[i - indexDiff]))
    return total


# ---------- Bài 6 ----------
def answer_min_abs_diff_with_gap(nums, x):
    sorted_list, best = [], float("inf")
    for i in range(x, len(nums)):
        bisect.insort(sorted_list, nums[i - x])
        pos = bisect.bisect_left(sorted_list, nums[i])
        if pos < len(sorted_list):
            best = min(best, sorted_list[pos] - nums[i])
        if pos > 0:
            best = min(best, nums[i] - sorted_list[pos - 1])
    return best


assert answer_almost_dup_bisect([1, 2, 3, 1], 3, 0) == True
assert answer_almost_dup_bisect([1, 5, 9, 1, 5, 9], 2, 3) == False
assert answer_almost_dup_bisect([8, -4, 3, 9, -2, 0], 2, 2) == True
assert answer_almost_dup_bisect([1, 3], 1, 2) == True
assert answer_find_almost_dup_pair([1, 2, 3, 1], 3, 0) == (0, 3)
assert answer_find_almost_dup_pair([8, -4, 3, 9, -2, 0], 2, 2) == (4, 5)
assert answer_find_almost_dup_pair([1, 5, 9, 1, 5, 9], 2, 3) is None
assert answer_count_almost_dup_pairs([1, 2, 3, 1], 3, 1) == 4
assert answer_count_almost_dup_pairs([1, 5, 9, 1, 5, 9], 2, 3) == 0
assert answer_count_almost_dup_pairs([7, 7, 7], 2, 0) == 3
assert answer_min_abs_diff_with_gap([4, 3, 2, 4], 2) == 0
assert answer_min_abs_diff_with_gap([5, 3, 2, 10, 15], 1) == 1
assert answer_min_abs_diff_with_gap([1, 2, 3, 4], 3) == 3

# Test chéo với brute force trên input ngẫu nhiên
random.seed(2817)
for _ in range(2000):
    n = random.randint(2, 10)
    arr = [random.randint(-10, 10) for _ in range(n)]
    k, t = random.randint(1, n), random.randint(0, 6)
    brute_pairs = [(j, i) for i in range(n) for j in range(max(0, i - k), i) if abs(arr[i] - arr[j]) <= t]
    assert answer_almost_dup_bisect(arr, k, t) == (len(brute_pairs) > 0)
    pair = answer_find_almost_dup_pair(arr, k, t)
    if pair is None:
        assert not brute_pairs
    else:
        j, i = pair
        assert j < i and i - j <= k and abs(arr[i] - arr[j]) <= t
        assert i == min(p[1] for p in brute_pairs)          # i nhỏ nhất có thể
    assert answer_count_almost_dup_pairs(arr, k, t) == len(brute_pairs)
    g = random.randint(0, n - 1)
    brute_min = min(abs(arr[a] - arr[b]) for a in range(n) for b in range(n) if abs(a - b) >= g)
    assert answer_min_abs_diff_with_gap([v + 11 for v in arr], g) == brute_min   # +11 để nums[i] >= 1
print("Đáp án Bài 1 → 6 PASS toàn bộ assert và 2000 test chéo với brute force!")

[2, 1, 0, 0, -1, -1, -2, -3] -4 -3 False
nums=[4, 11, -3, 6, 0], k=2, t=3, w=t+1=4
i=0 x= 4 b= 1 | trước {}             | 3 thùng b-1, b, b+1 đều trống
      thêm 1:4 -> {1:4}
i=1 x=11 b= 2 | trước {1:4}          | thùng 1 có 4: |11-4|=7>3
      thêm 2:11 -> {1:4, 2:11}
i=2 x=-3 b=-1 | trước {1:4, 2:11}    | 3 thùng b-1, b, b+1 đều trống
      thêm -1:-3, xóa nums[0]=4 (thùng 1) -> {-1:-3, 2:11}
i=3 x= 6 b= 1 | trước {-1:-3, 2:11}  | thùng 2 có 11: |6-11|=5>3
      thêm 1:6, xóa nums[1]=11 (thùng 2) -> {-1:-3, 1:6}
i=4 x= 0 b= 0 | trước {-1:-3, 1:6}   | thùng -1 có -3: |0-(-3)|=3<=3 -> True

Đáp án Bài 1 → 6 PASS toàn bộ assert và 2000 test chéo với brute force!


## 19. Template cuối cùng nên nhớ

**Bucket (nộp bài, `O(n)`)** — nhớ 3 câu: *thùng rộng `t + 1`, id = `x // w`; xét `b - 1, b, b + 1`; kiểm tra → thêm → xóa `nums[i - k]`.*

```python
class Solution:
    def containsNearbyAlmostDuplicate(self, nums: List[int], indexDiff: int, valueDiff: int) -> bool:
        w = valueDiff + 1
        buckets = {}
        for i, x in enumerate(nums):
            b = x // w
            if b in buckets:
                return True
            if b - 1 in buckets and x - buckets[b - 1] <= valueDiff:
                return True
            if b + 1 in buckets and buckets[b + 1] - x <= valueDiff:
                return True
            buckets[b] = x
            if i >= indexDiff:
                del buckets[nums[i - indexDiff] // w]
        return False
```

**Cửa sổ sắp xếp + `bisect` (dùng lại được cho bài đếm / tìm min)** — *hỏi ceiling của `x - t` → `insort(x)` → bỏ `nums[i - k]`.*

```python
window = []
for i, x in enumerate(nums):
    pos = bisect.bisect_left(window, x - t)
    if pos < len(window) and window[pos] <= x + t:
        return True
    bisect.insort(window, x)
    if i >= k:
        window.pop(bisect.bisect_left(window, nums[i - k]))
return False
```

## 20. Checklist trước khi submit

- [ ] Dùng đúng tên method `containsNearbyAlmostDuplicate` và tham số `indexDiff`, `valueDiff`?
- [ ] Độ rộng thùng là `valueDiff + 1` (không phải `valueDiff`) — an toàn khi `valueDiff = 0`?
- [ ] Id thùng tính bằng `x // w` (làm tròn xuống), **không** dùng `int(x / w)`? (Nếu viết C++/Java: đã xử lý số âm và dùng `long`?)
- [ ] Đã xét **đủ 3 thùng** `b - 1`, `b`, `b + 1`, và với thùng kề đã **so thật** `<= valueDiff`?
- [ ] Thứ tự mỗi bước là **kiểm tra → thêm → xóa**, để không tự so với chính mình và không bỏ sót cặp cách đúng `indexDiff`?
- [ ] Xóa đúng phần tử `nums[i - indexDiff]` và chỉ khi `i >= indexDiff`?
- [ ] Dùng `<=` (không phải `<`) cho cả hai điều kiện?
- [ ] Đã thử: hai ví dụ của đề, số âm, `valueDiff = 0`, `indexDiff = n`, cặp ở khoảng cách **đúng** `indexDiff`, giá trị biên `±10^9`?
- [ ] Nếu dùng Cách 2: tìm ceiling của **`x - t`** (không phải của `x`) và kiểm tra `pos < len(window)` trước khi đọc `window[pos]`?